# LUDB-basiertes Retuning des Delineation-Algorithmus

Nutzt die vollstaendige LUDB-Ground-Truth (200 Records x 12 Leads), um die Fenstergrenzen und Konsens-Feature-Strategien des in `ludb_validation.ipynb` validierten Algorithmus (`vcgsuite.annotation.hierarchical`) neu zu kalibrieren -- orientiert an `FirstNotebookForTraining.ipynb` (Zelle 5-8: deterministisches Tuning + RF-Hybrid), aber mit der aufgeraeumten `vcgsuite`-Bibliothek statt Notebook-Code, und mit echtem Train/Test-Split statt Training+Evaluation auf denselben 22 Beats wie im Original.

**Ausgangslage (siehe `baseline_ludb_metrics.csv`, Momentaufnahme aus `ludb_validation.ipynb` Schritt 4):** QRS-Marker bereits gut (F1 ~89%), P-Marker mittelmaessig (F1 61-68%), T_on/T_peak brauchbar (F1 ~73-75%), **T_off praktisch kaputt (F1 = 11.65%)**.

**Vereinbarter Umfang:** alle 12 Marker adressieren (7 mit direkter LUDB-Entsprechung werden per Grid-Search ueber Fenster + Feature-Strategie neu kalibriert; die uebrigen 5 ohne direkte Entsprechung werden transparent behandelt, siehe unten), Record-Level Cross-Validation fuer die Auswahl, deterministisches Retuning **und** RF-Hybrid, Notebook-first (Uebernahme in `vcgsuite` erst nach Bestaetigung).

**Umfang dieses Notebooks (Teil 1 von 2):** deterministisches Fenster-/Feature-Retuning fuer die 7 direkt supervisierten Marker, Evaluation auf echtem Test-Holdout. Der RF-Hybrid-Teil (Teil 2, analog zu Zelle 7/8 im Original) baut auf den hier gecachten Trainingsdaten auf und folgt als naechster Schritt.

Verwendet gemeinsame Hilfsfunktionen aus `ludb_common.py` (Kopie der in `ludb_validation.ipynb` validierten Parsing-/Integrations-/Metrik-Funktionen -- `ludb_validation.ipynb` selbst bleibt unveraendert).

In [ ]:
from __future__ import annotations

import contextlib
import io
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import wfdb
import plotly.graph_objects as go

import vcgsuite as ecg
from vcgsuite.annotation.features import extract_features, detect_consensus
from vcgsuite.kinematics.constants import HIERARCHICAL_WINDOWS, FEATURE_COLS, Q_OFF_OFFSET_S

import ludb_common as lc

pd.set_option("display.max_columns", 20)

DATA_DIR = lc.DATA_DIR
LEADS = ["i", "ii", "iii", "avr", "avl", "avf", "v1", "v2", "v3", "v4", "v5", "v6"]
HW = HIERARCHICAL_WINDOWS  # bestehende Fenstergrenzen als Ausgangspunkt fuer die Suche
CANDIDATE_FEATURES = [c for c in FEATURE_COLS if c != "rel_t_ms"]
print(f"{len(CANDIDATE_FEATURES)} Kandidaten-Features x 2 (MAX/MIN) = {len(CANDIDATE_FEATURES)*2} Kandidaten pro Fenster.")

## Baseline (Momentaufnahme)

Referenzwerte aus `ludb_validation.ipynb` Schritt 4, vor jedem Tuning.

In [ ]:
df_baseline = pd.read_csv("baseline_ludb_metrics.csv", index_col="Wave")
df_baseline.round(2)

## Train/Test-Split (Record-Level)

80/20-Split nach Record-ID, nicht nach Beat — sonst koennten Beats desselben Records (aehnliche Morphologie, gleicher Proband) in Train UND Test landen, was die Verbesserung optimistisch verzerren wuerde. Der Test-Split wird bis zur finalen Evaluation ganz unten nicht angeruehrt. Innerhalb des Train-Splits wird zusaetzlich in 5 Folds aufgeteilt, damit die Grid-Search-Rangfolge (welches Fenster/Feature gewinnt) ueber mehrere Teilmengen gemittelt wird, statt nur einer einzigen Aufteilung zu vertrauen (Record-Level Cross-Validation).

In [ ]:
N_FOLDS = 5
TEST_FRACTION = 0.2
RANDOM_SEED = 42

all_record_ids = sorted(int(p.stem) for p in DATA_DIR.glob("*.hea"))
rng = random.Random(RANDOM_SEED)
shuffled = all_record_ids[:]
rng.shuffle(shuffled)

n_test = int(round(len(shuffled) * TEST_FRACTION))
test_ids = sorted(shuffled[:n_test])
train_ids = sorted(shuffled[n_test:])

print(f"Train: {len(train_ids)} Records ({N_FOLDS}-fach gefaltet fuer die Grid-Search)")
print(f"Test:  {len(test_ids)} Records (Holdout, unangetastet bis zur finalen Evaluation)")

## Effiziente Grid-Search-Infrastruktur

Eine Grid-Search ueber ~60 Feature/Op-Kandidaten x mehrere Fenstergrenzen pro Marker waere unbezahlbar, wenn `extract_features()` (ruft Glaettung, Gradienten, Rang-Normierung etc. auf) fuer jeden Kandidaten neu aufgerufen wuerde. Stattdessen: pro Beat wird `extract_features()` **einmal** ueber ein breites Fenster (±220 ms um den Anker) aufgerufen — unveraendert aus `vcgsuite.annotation.features` importiert, keine eigene Reimplementierung. Alle Fenster-/Feature-Kandidaten werden danach per einfachem Masking/`argmax` aus diesem bereits berechneten DataFrame bedient.

In [ ]:
def wide_feature_window(df_analysis: pd.DataFrame, anchor_t: float, half_width_ms: float = 220.0):
    """Extrahiert einmalig ein breites Feature-Fenster um einen Anker-
    Zeitpunkt (Basis fuer die Grid-Search unten). Ruft nur die
    unveraenderte vcgsuite.annotation.features.extract_features() auf."""
    if anchor_t is None or np.isnan(anchor_t):
        return None
    t_all = df_analysis["Time"].values
    tlo, thi = anchor_t - half_width_ms / 1000.0, anchor_t + half_width_ms / 1000.0
    twin = t_all[(t_all >= tlo) & (t_all <= thi)]
    if len(twin) < 3:
        return None
    return extract_features(df_analysis, twin, anchor_t)


def predict_from_feat(feat, lo_ms: float, hi_ms: float, fcol: str, op: str) -> float:
    """Wendet einen (Fenster, Feature, MAX/MIN)-Kandidaten auf ein bereits
    extrahiertes breites Feature-Fenster an -> Zeitpunkt [s] oder NaN.
    Reine Slicing-Operation, kein erneuter extract_features()-Call."""
    if feat is None:
        return np.nan
    sub = feat[(feat["rel_t_ms"] >= lo_ms) & (feat["rel_t_ms"] <= hi_ms)]
    if len(sub) == 0 or fcol not in sub.columns:
        return np.nan
    vals = sub[fcol].values.astype(float)
    if len(vals) == 0 or np.nanstd(vals) < 1e-10:
        return np.nan
    idx = int(np.nanargmax(vals)) if op == "MAX" else int(np.nanargmin(vals))
    return float(sub["t_abs"].values[idx])


def predict_consensus_from_feat(feat, lo_ms: float, hi_ms: float, strategies: list) -> float:
    """Median-Konsens ueber mehrere (Feature, Op)-Strategien, angewandt auf
    ein bereits extrahiertes breites Feature-Fenster -- Aequivalent zu
    detect_consensus(), nur ohne erneute extract_features()-Aufrufe."""
    if feat is None:
        return np.nan
    votes = [predict_from_feat(feat, lo_ms, hi_ms, fcol, op) for fcol, op in strategies]
    votes = [v for v in votes if not np.isnan(v)]
    return float(np.median(votes)) if votes else np.nan


def nearest_gt(pooled_samples: list, approx_t: float, fs: float, max_dist_s: float = 0.15) -> float:
    """Naechster (ueber alle 12 Leads gepoolter) GT-Sample zu einem
    ungefaehren Zeitpunkt (z.B. der aktuellen, ungetunten Algorithmus-
    Detektion), als Trainingsziel fuer diesen Beat. ACHTUNG: das ist eine
    andere GT-Zuordnung als in Schritt 4 (dort bewusst pro Lead getrennt,
    fuer eine einzelne Trainings-Zielgroesse pro Beat reicht die gepoolte
    naechstgelegene Annotation)."""
    if not pooled_samples or np.isnan(approx_t):
        return np.nan
    approx_sample = approx_t * fs
    arr = np.asarray(pooled_samples)
    idx = int(np.argmin(np.abs(arr - approx_sample)))
    if abs(arr[idx] - approx_sample) / fs > max_dist_s:
        return np.nan
    return float(arr[idx]) / fs

## Trainings-Cache aufbauen

Einmaliger Lauf ueber alle Trainings-Records: Signal filtern -> Frank-XYZ -> Kinematik -> R-Peak/-Turn -> Standard-Annotation (aus `ludb_common.run_delineation_on_ludb_record`, unveraendert). `df_analysis` wird pro Record im Speicher gehalten (`cache`), damit spaeter fuer die kaskadierten Marker (P_on/P_off, T_on) neue Feature-Fenster um *neu berechnete* Anker extrahiert werden koennen, ohne die Filter-/Transformations-Pipeline erneut zu durchlaufen (das waere der teure Teil). Gleichzeitig wird pro Beat das ueber alle 12 Leads gepoolte Trainingsziel (`gt_<marker>`) bestimmt.

**Laufzeithinweis:** ~160 Records durch die volle Pipeline (Filter/Kinematik/Annotation) plus 2400+ `extract_features()`-Aufrufe weiter unten — je nach Rechenleistung mehrere Minuten.

In [ ]:
GT_MARKERS = ["QRS_on", "P_peak", "QRS_off", "P_on", "P_off", "T_on", "T_off"]

cache: dict[int, tuple[pd.DataFrame, pd.DataFrame]] = {}
beat_rows = []

t0 = time.time()
for i, rid in enumerate(train_ids):
    detections, df_beats, df_analysis = lc.run_delineation_on_ludb_record(rid, verbose=False)
    cache[rid] = (df_analysis, df_beats)
    fs = df_analysis.attrs["fs"]
    fold = i % N_FOLDS

    pooled = {w: [] for w in GT_MARKERS}
    for lead in LEADS:
        gt_lead = lc.parse_ludb_annotations(rid, lead)
        for w in GT_MARKERS:
            pooled[w].extend(gt_lead[w])
    for w in GT_MARKERS:
        pooled[w].sort()

    for _, row in df_beats.iterrows():
        r_peak_t = row["t_R_peak+"]
        if np.isnan(r_peak_t):
            continue
        entry = {
            "record_id": rid, "beat_id": int(row["beat_id"]), "fold": fold,
            "r_peak_t": r_peak_t,
            "cur_Q_on": row["t_Q_on"], "cur_S_off": row["t_S_off"], "cur_P_peak": row["t_P_peak"],
            "cur_P_on": row["t_P_on"], "cur_P_off": row["t_P_off"],
            "cur_T_on": row["t_T_on"], "cur_T_turn2": row["t_T_turn2"], "cur_T_off": row["t_T_off"],
        }
        entry["gt_QRS_on"]  = nearest_gt(pooled["QRS_on"],  row["t_Q_on"],   fs)
        entry["gt_P_peak"]  = nearest_gt(pooled["P_peak"],  row["t_P_peak"], fs)
        entry["gt_QRS_off"] = nearest_gt(pooled["QRS_off"], row["t_S_off"],  fs)
        entry["gt_P_on"]    = nearest_gt(pooled["P_on"],    row["t_P_on"],   fs)
        entry["gt_P_off"]   = nearest_gt(pooled["P_off"],   row["t_P_off"],  fs)
        entry["gt_T_on"]    = nearest_gt(pooled["T_on"],    row["t_T_on"],   fs)
        entry["gt_T_off"]   = nearest_gt(pooled["T_off"],   row["t_T_off"],  fs)
        beat_rows.append(entry)

    if (i + 1) % 20 == 0:
        print(f"  {i + 1}/{len(train_ids)} Trainings-Records verarbeitet ({time.time() - t0:.0f}s)")

df_beats_train = pd.DataFrame(beat_rows)
print(f"\n{len(df_beats_train)} Beats aus {len(cache)} Trainings-Records gecacht "
      f"({time.time() - t0:.0f}s gesamt).")
print("\nGT gefunden (von Beats mit R_peak):")
for w in GT_MARKERS:
    print(f"  gt_{w:8s}: {df_beats_train[f'gt_{w}'].notna().sum():5d} / {len(df_beats_train)}")

## Grid-Search-Funktionen

`tune_marker()` lauft dreistufig pro Marker:

1. **Feature-Ranking im Basisfenster** (den aktuellen `HIERARCHICAL_WINDOWS`-Grenzen): alle ~60 (Feature, MAX/MIN)-Kandidaten gegen das GT bewertet, per Record-Level-CV (Rangfolge wird pro Trainings-Fold einzeln berechnet und gemittelt, nicht nur global).
2. **Fenstergrenzen-Suche** mit dem besten Einzelfeature aus Schritt 1: kleine Verschiebungs-Kombinationen (±20/±40 ms) um die Basisgrenzen.
3. **Finales Feature-Ranking** im (ggf. verschobenen) Fenster -> die Top-3-Features bilden die neue Konsens-Strategie (Median-Vote, wie `detect_consensus()` es bereits tut).

Bewertungskriterium: Match-Rate (Anteil Beats mit Vorhersagefehler ≤75 ms, gleiche Toleranz wie in Schritt 4) zuerst, mittlerer absoluter Fehler (MAE) als Tie-Breaker.

In [ ]:
def _score_stats(errors: np.ndarray, tol_ms: float = 75.0):
    valid = errors[~np.isnan(errors)]
    within = valid[np.abs(valid) <= tol_ms]
    match_rate = len(within) / len(errors) if len(errors) else np.nan
    mae = float(np.mean(np.abs(within))) if len(within) else np.nan
    return match_rate, mae


def _cv_aggregate(errors: np.ndarray, folds: np.ndarray, tol_ms: float = 75.0):
    rates, maes = [], []
    for f in np.unique(folds):
        mr, mae = _score_stats(errors[folds == f], tol_ms)
        if not np.isnan(mr):
            rates.append(mr)
        if not np.isnan(mae):
            maes.append(mae)
    return (float(np.mean(rates)) if rates else np.nan,
            float(np.mean(maes)) if maes else np.nan)


def score_candidate(wide_feats: list, gt_rel_ms: np.ndarray, lo_ms: float, hi_ms: float,
                    fcol: str, op: str) -> np.ndarray:
    """Vorhersagefehler (Vorhersage - GT) [ms] je Beat fuer einen
    (Fenster, Feature, Op)-Kandidaten."""
    n = len(wide_feats)
    errors = np.full(n, np.nan)
    for i in range(n):
        feat, gt = wide_feats[i], gt_rel_ms[i]
        if feat is None or np.isnan(gt):
            continue
        sub = feat[(feat["rel_t_ms"] >= lo_ms) & (feat["rel_t_ms"] <= hi_ms)]
        if len(sub) == 0 or fcol not in sub.columns:
            continue
        vals = sub[fcol].values.astype(float)
        if np.nanstd(vals) < 1e-10:
            continue
        idx = int(np.nanargmax(vals)) if op == "MAX" else int(np.nanargmin(vals))
        errors[i] = sub["rel_t_ms"].values[idx] - gt
    return errors


def rank_features_at_window(wide_feats: list, gt_rel_ms: np.ndarray, folds: np.ndarray,
                            lo_ms: float, hi_ms: float, candidate_features: list,
                            tol_ms: float = 75.0) -> pd.DataFrame:
    """Bewertet alle (Feature, MAX/MIN)-Kandidaten in EINEM festen Fenster.
    Pro Beat wird nur einmal auf [lo_ms,hi_ms] geslict, alle Feature-
    Spalten werden aus diesem Sub-Frame gelesen statt pro Kandidat neu zu
    slicen -- haelt die Suche schnell genug fuer ~60 Kandidaten."""
    n = len(wide_feats)
    subs = []
    for feat in wide_feats:
        if feat is None:
            subs.append(None); continue
        sub = feat[(feat["rel_t_ms"] >= lo_ms) & (feat["rel_t_ms"] <= hi_ms)]
        subs.append(sub if len(sub) > 0 else None)

    results = []
    for fcol in candidate_features:
        for op in ("MAX", "MIN"):
            errors = np.full(n, np.nan)
            for i, sub in enumerate(subs):
                gt = gt_rel_ms[i]
                if sub is None or np.isnan(gt) or fcol not in sub.columns:
                    continue
                vals = sub[fcol].values.astype(float)
                if np.nanstd(vals) < 1e-10:
                    continue
                idx = int(np.nanargmax(vals)) if op == "MAX" else int(np.nanargmin(vals))
                errors[i] = sub["rel_t_ms"].values[idx] - gt
            cv_mr, cv_mae = _cv_aggregate(errors, folds, tol_ms)
            results.append({"fcol": fcol, "op": op, "cv_match_rate": cv_mr, "cv_mae_ms": cv_mae})
    df = pd.DataFrame(results)
    return df.sort_values(["cv_match_rate", "cv_mae_ms"], ascending=[False, True]).reset_index(drop=True)


def rank_windows_for_feature(wide_feats: list, gt_rel_ms: np.ndarray, folds: np.ndarray,
                             window_candidates: list, fcol: str, op: str,
                             tol_ms: float = 75.0) -> pd.DataFrame:
    results = []
    for lo_ms, hi_ms in window_candidates:
        errors = score_candidate(wide_feats, gt_rel_ms, lo_ms, hi_ms, fcol, op)
        cv_mr, cv_mae = _cv_aggregate(errors, folds, tol_ms)
        results.append({"lo_ms": lo_ms, "hi_ms": hi_ms, "cv_match_rate": cv_mr, "cv_mae_ms": cv_mae})
    df = pd.DataFrame(results)
    return df.sort_values(["cv_match_rate", "cv_mae_ms"], ascending=[False, True]).reset_index(drop=True)


def tune_marker(wide_feats: list, gt_rel_ms: np.ndarray, folds: np.ndarray,
                base_lo_ms: float, base_hi_ms: float, candidate_features: list,
                window_shifts=(-40, -20, 0, 20, 40), top_k: int = 3, tol_ms: float = 75.0) -> dict:
    df1 = rank_features_at_window(wide_feats, gt_rel_ms, folds, base_lo_ms, base_hi_ms, candidate_features, tol_ms)
    best_fcol, best_op = df1.iloc[0]["fcol"], df1.iloc[0]["op"]
    baseline_row = df1.iloc[0]

    window_candidates = sorted({
        (base_lo_ms + dlo, base_hi_ms + dhi)
        for dlo in window_shifts for dhi in window_shifts
        if (base_hi_ms + dhi) - (base_lo_ms + dlo) >= 20
    })
    df2 = rank_windows_for_feature(wide_feats, gt_rel_ms, folds, window_candidates, best_fcol, best_op, tol_ms)
    new_lo, new_hi = float(df2.iloc[0]["lo_ms"]), float(df2.iloc[0]["hi_ms"])

    df3 = rank_features_at_window(wide_feats, gt_rel_ms, folds, new_lo, new_hi, candidate_features, tol_ms)
    top = df3.head(top_k)
    strategies = list(zip(top["fcol"], top["op"]))

    return {
        "lo_ms": new_lo, "hi_ms": new_hi, "strategies": strategies,
        "cv_match_rate": float(top.iloc[0]["cv_match_rate"]), "cv_mae_ms": float(top.iloc[0]["cv_mae_ms"]),
        "baseline_lo_ms": base_lo_ms, "baseline_hi_ms": base_hi_ms,
        "baseline_feature": (best_fcol, best_op),
        "baseline_match_rate": float(baseline_row["cv_match_rate"]), "baseline_mae_ms": float(baseline_row["cv_mae_ms"]),
    }

## Pass 1 — R_peak-verankerte Marker (Q_on, S_off, P_peak)

Diese drei Marker haengen direkt an `R_peak+` (externer Anker, nicht selbst getunt). Ein einziges breites Feature-Fenster pro Beat reicht fuer alle drei.

In [ ]:
HALF_WIDTH_MS = 220.0

t0 = time.time()
wide_feats_rpeak = [
    wide_feature_window(cache[rid][0], r_peak_t, HALF_WIDTH_MS)
    for rid, r_peak_t in zip(df_beats_train["record_id"], df_beats_train["r_peak_t"])
]
folds_arr = df_beats_train["fold"].values
print(f"{sum(f is not None for f in wide_feats_rpeak)}/{len(wide_feats_rpeak)} "
      f"breite Feature-Fenster extrahiert ({time.time() - t0:.0f}s).")

In [ ]:
gt_rel_Q_on  = (df_beats_train["gt_QRS_on"].values  - df_beats_train["r_peak_t"].values) * 1000
gt_rel_S_off = (df_beats_train["gt_QRS_off"].values - df_beats_train["r_peak_t"].values) * 1000
gt_rel_Ppeak = (df_beats_train["gt_P_peak"].values  - df_beats_train["r_peak_t"].values) * 1000

t0 = time.time()
tuned_Q_on  = tune_marker(wide_feats_rpeak, gt_rel_Q_on,  folds_arr, HW["Q_on"]["lo_ms"],   HW["Q_on"]["hi_ms"],   CANDIDATE_FEATURES)
tuned_S_off = tune_marker(wide_feats_rpeak, gt_rel_S_off, folds_arr, HW["S_off"]["lo_ms"],  HW["S_off"]["hi_ms"],  CANDIDATE_FEATURES)
tuned_Ppeak = tune_marker(wide_feats_rpeak, gt_rel_Ppeak, folds_arr, HW["P_peak"]["lo_ms"], HW["P_peak"]["hi_ms"], CANDIDATE_FEATURES)
print("Pass 1 fertig ({:.0f}s).".format(time.time() - t0))

def _win_str(lo, hi):
    return "{:+.0f}/{:+.0f}ms".format(lo, hi)

pass1_summary = pd.DataFrame([
    {"Marker": "Q_on",   "Fenster (alt)": _win_str(HW["Q_on"]["lo_ms"], HW["Q_on"]["hi_ms"]),
     "Fenster (neu)": _win_str(tuned_Q_on["lo_ms"], tuned_Q_on["hi_ms"]),
     "Top-Feature": tuned_Q_on["strategies"][0], "CV Match-Rate": tuned_Q_on["cv_match_rate"], "CV MAE (ms)": tuned_Q_on["cv_mae_ms"]},
    {"Marker": "S_off",  "Fenster (alt)": _win_str(HW["S_off"]["lo_ms"], HW["S_off"]["hi_ms"]),
     "Fenster (neu)": _win_str(tuned_S_off["lo_ms"], tuned_S_off["hi_ms"]),
     "Top-Feature": tuned_S_off["strategies"][0], "CV Match-Rate": tuned_S_off["cv_match_rate"], "CV MAE (ms)": tuned_S_off["cv_mae_ms"]},
    {"Marker": "P_peak", "Fenster (alt)": _win_str(HW["P_peak"]["lo_ms"], HW["P_peak"]["hi_ms"]),
     "Fenster (neu)": _win_str(tuned_Ppeak["lo_ms"], tuned_Ppeak["hi_ms"]),
     "Top-Feature": tuned_Ppeak["strategies"][0], "CV Match-Rate": tuned_Ppeak["cv_match_rate"], "CV MAE (ms)": tuned_Ppeak["cv_mae_ms"]},
]).set_index("Marker")
pass1_summary.round(3)

## Pass 2 — kaskadierte Marker (P_on/P_off auf P_peak, T_on auf S_off)

P_on/P_off und T_on haengen nicht an R_peak+, sondern an P_peak bzw. S_off — hier werden die in Pass 1 **neu getunten** P_peak-/S_off-Zeitpunkte je Beat als Anker verwendet (Median-Konsens ueber die neuen Top-3-Strategien, wiederverwendet aus dem bereits gecachten Pass-1-Feature-Fenster, da P_peak/S_off innerhalb der ±220ms-Breite liegen). Fuer P_on/P_off/T_on selbst wird dann je ein frisches, um den jeweils neuen Anker zentriertes Feature-Fenster extrahiert.

In [ ]:
new_P_peak_t = np.array([
    predict_consensus_from_feat(feat, tuned_Ppeak["lo_ms"], tuned_Ppeak["hi_ms"], tuned_Ppeak["strategies"])
    for feat in wide_feats_rpeak
])
new_S_off_t = np.array([
    predict_consensus_from_feat(feat, tuned_S_off["lo_ms"], tuned_S_off["hi_ms"], tuned_S_off["strategies"])
    for feat in wide_feats_rpeak
])
print(f"Neue P_peak-Anker: {np.sum(~np.isnan(new_P_peak_t))}/{len(new_P_peak_t)}")
print(f"Neue S_off-Anker:  {np.sum(~np.isnan(new_S_off_t))}/{len(new_S_off_t)}")

In [ ]:
t0 = time.time()
wide_feats_ppeak = [
    wide_feature_window(cache[rid][0], anchor_t, HALF_WIDTH_MS)
    for rid, anchor_t in zip(df_beats_train["record_id"], new_P_peak_t)
]
wide_feats_soff = [
    wide_feature_window(cache[rid][0], anchor_t, HALF_WIDTH_MS)
    for rid, anchor_t in zip(df_beats_train["record_id"], new_S_off_t)
]
print("Frische Feature-Fenster extrahiert ({:.0f}s).".format(time.time() - t0))

gt_rel_P_on  = (df_beats_train["gt_P_on"].values  - new_P_peak_t) * 1000
gt_rel_P_off = (df_beats_train["gt_P_off"].values - new_P_peak_t) * 1000
gt_rel_T_on  = (df_beats_train["gt_T_on"].values  - new_S_off_t) * 1000

t0 = time.time()
tuned_P_on  = tune_marker(wide_feats_ppeak, gt_rel_P_on,  folds_arr, HW["P_on"]["lo_ms"],  HW["P_on"]["hi_ms"],  CANDIDATE_FEATURES)
tuned_P_off = tune_marker(wide_feats_ppeak, gt_rel_P_off, folds_arr, HW["P_off"]["lo_ms"], HW["P_off"]["hi_ms"], CANDIDATE_FEATURES)
tuned_T_on  = tune_marker(wide_feats_soff,  gt_rel_T_on,  folds_arr, HW["T_on"]["lo_ms"],  HW["T_on"]["hi_ms"],  CANDIDATE_FEATURES)
print("Pass 2 fertig ({:.0f}s).".format(time.time() - t0))

def _win_str(lo, hi):
    return "{:+.0f}/{:+.0f}ms".format(lo, hi)

pass2_summary = pd.DataFrame([
    {"Marker": "P_on",  "Fenster (alt)": _win_str(HW["P_on"]["lo_ms"], HW["P_on"]["hi_ms"]),
     "Fenster (neu)": _win_str(tuned_P_on["lo_ms"], tuned_P_on["hi_ms"]),
     "Top-Feature": tuned_P_on["strategies"][0], "CV Match-Rate": tuned_P_on["cv_match_rate"], "CV MAE (ms)": tuned_P_on["cv_mae_ms"]},
    {"Marker": "P_off", "Fenster (alt)": _win_str(HW["P_off"]["lo_ms"], HW["P_off"]["hi_ms"]),
     "Fenster (neu)": _win_str(tuned_P_off["lo_ms"], tuned_P_off["hi_ms"]),
     "Top-Feature": tuned_P_off["strategies"][0], "CV Match-Rate": tuned_P_off["cv_match_rate"], "CV MAE (ms)": tuned_P_off["cv_mae_ms"]},
    {"Marker": "T_on",  "Fenster (alt)": _win_str(HW["T_on"]["lo_ms"], HW["T_on"]["hi_ms"]),
     "Fenster (neu)": _win_str(tuned_T_on["lo_ms"], tuned_T_on["hi_ms"]),
     "Top-Feature": tuned_T_on["strategies"][0], "CV Match-Rate": tuned_T_on["cv_match_rate"], "CV MAE (ms)": tuned_T_on["cv_mae_ms"]},
]).set_index("Marker")
pass2_summary.round(3)

## Pass 3 — T_off (Prioritaet 1: F1 = 11.65% in der Baseline)

T_off haengt in der aktuellen Architektur an `T_turn2`, das Ende einer dreistufigen Kaskade (`S_off -> T_on -> T_turn1 -> T_turn2 -> T_off`) ohne eigenes LUDB-GT fuer die Zwischenstationen T_turn1/T_turn2. Der katastrophale F1-Wert koennte also entweder an T_offs eigenem Fenster/Feature liegen, oder daran, dass T_turn2 selbst schon an der falschen Stelle sitzt (Fehler-Kaskade). Deshalb zwei Varianten:

- **Variante A** (Architektur erhalten): T_off wird wie bisher relativ zu T_turn2 gesucht, nur Fenster/Feature neu getunt.
- **Variante B** (Architektur-Experiment): T_off wird **direkt** relativ zu T_on gesucht, T_turn1/T_turn2 werden uebersprungen.

Beide werden per CV bewertet, die bessere wird fuer die finale Evaluation uebernommen — transparent, keine der beiden wird stillschweigend bevorzugt.

In [ ]:
new_T_on_t = np.array([
    predict_consensus_from_feat(feat, tuned_T_on["lo_ms"], tuned_T_on["hi_ms"], tuned_T_on["strategies"])
    for feat in wide_feats_soff
])

t0 = time.time()
wide_feats_Tturn2_current = [
    wide_feature_window(cache[rid][0], cur_t, HALF_WIDTH_MS)
    for rid, cur_t in zip(df_beats_train["record_id"], df_beats_train["cur_T_turn2"])
]
wide_feats_Ton_new = [
    wide_feature_window(cache[rid][0], anchor_t, 350.0)  # T_on -> T_off direkt: groesserer Suchradius
    for rid, anchor_t in zip(df_beats_train["record_id"], new_T_on_t)
]
print("Feature-Fenster fuer beide T_off-Varianten extrahiert ({:.0f}s).".format(time.time() - t0))

gt_rel_T_off_A = (df_beats_train["gt_T_off"].values - df_beats_train["cur_T_turn2"].values) * 1000
gt_rel_T_off_B = (df_beats_train["gt_T_off"].values - new_T_on_t) * 1000

t0 = time.time()
tuned_T_off_A = tune_marker(wide_feats_Tturn2_current, gt_rel_T_off_A, folds_arr,
                            HW["T_off"]["lo_ms"], HW["T_off"]["hi_ms"], CANDIDATE_FEATURES)
tuned_T_off_B = tune_marker(wide_feats_Ton_new, gt_rel_T_off_B, folds_arr,
                            130, 410, CANDIDATE_FEATURES, window_shifts=(-60, -30, 0, 30, 60))
print("Pass 3 fertig ({:.0f}s).".format(time.time() - t0))

def _win_str(lo, hi):
    return "{:+.0f}/{:+.0f}ms".format(lo, hi)

t_off_compare = pd.DataFrame([
    {"Variante": "A (via T_turn2, aktuelle Architektur)", "Fenster": _win_str(tuned_T_off_A["lo_ms"], tuned_T_off_A["hi_ms"]),
     "Top-Feature": tuned_T_off_A["strategies"][0], "CV Match-Rate": tuned_T_off_A["cv_match_rate"], "CV MAE (ms)": tuned_T_off_A["cv_mae_ms"]},
    {"Variante": "B (direkt via T_on)", "Fenster": _win_str(tuned_T_off_B["lo_ms"], tuned_T_off_B["hi_ms"]),
     "Top-Feature": tuned_T_off_B["strategies"][0], "CV Match-Rate": tuned_T_off_B["cv_match_rate"], "CV MAE (ms)": tuned_T_off_B["cv_mae_ms"]},
]).set_index("Variante")
t_off_compare.round(3)

In [ ]:
T_OFF_VARIANT = "A" if tuned_T_off_A["cv_match_rate"] >= tuned_T_off_B["cv_match_rate"] else "B"
tuned_T_off = tuned_T_off_A if T_OFF_VARIANT == "A" else tuned_T_off_B
print(f"Gewaehlt: Variante {T_OFF_VARIANT} "
      f"(CV Match-Rate {tuned_T_off['cv_match_rate']:.1%} vs. Baseline-F1 11.65% in Schritt 4).")

### Pass 3b — T_off nachgebessert (engeres Fenster, ohne `*_lmax`-Features)

`ludb_projection_reconstruction.ipynb` hat unabhaengig von jeder menschlichen GT gezeigt: unser T_off
(Variante A, Fenster +20/+100ms nach T_turn2, Top-Feature `(r_lmax, MAX)`) liegt **+132.6ms (mean) / +131.0ms
(median), SD=48.5ms** spaeter als der Median von 8 rein geometrisch aus der 3D-Trajektorie rekonstruierten
Lead-Schaetzungen -- eng, unimodal, klar systematisch (kein Rauschen). 130ms nach dem T-Wellen-Ende liegt
genau im typischen T-U-Intervall -- `r_lmax` (rollierendes lokales Maximum von r) markiert vermutlich eine
U-Welle oder eine vergleichbare Sekundaerstruktur statt des echten T-Endes (siehe
`docs/ludb_projection_blindness_findings.md`, Abschnitt 4).

Fix: Pass 3 fuer T_off (Variante A) erneut laufen lassen, aber (a) `hi_ms` hart auf **<=80ms** begrenzt
(statt der bisherigen 100ms, die schon in die Sekundaerstruktur reichten) und (b) alle `*_lmax`-Features
aus dem Kandidatenpool ausgeschlossen. Nutzt dieselben bereits gecachten `wide_feats_Tturn2_current` /
`gt_rel_T_off_A` aus Pass 3 (kein erneutes Feature-Caching noetig, nur ein neuer, restriktiverer
Fenster-Grid-Search). **Ueberschreibt `tuned_T_off`** -- alle nachfolgenden Zellen (Zusammenfassung,
`tuned_config`, Test-Evaluation, RF-Hybrid, Teil 3/4) muessen danach neu ausgefuehrt werden, um den Fix zu
uebernehmen; sie lesen `tuned_T_off` per Variablenname, keine weiteren Codeaenderungen noetig.

In [ ]:
CANDIDATE_FEATURES_NO_LMAX = [f for f in CANDIDATE_FEATURES if not f.endswith("_lmax")]
print(f"{len(CANDIDATE_FEATURES)} -> {len(CANDIDATE_FEATURES_NO_LMAX)} Kandidaten-Features "
      f"(ausgeschlossen: {[f for f in CANDIDATE_FEATURES if f.endswith('_lmax')]}).")

# Restriktives Fenster-Grid: hi_ms wird hart auf <=80ms begrenzt (T_turn2 + 80ms statt + 140/100ms),
# um die vermutete Sekundaerstruktur (U-Welle) ausserhalb des Suchraums zu halten.
T_OFF_WINDOW_CANDIDATES = sorted({
    (lo, hi)
    for lo in (0, 10, 20, 30, 40, 50)
    for hi in (40, 50, 60, 70, 80)
    if hi - lo >= 20
})
print(f"{len(T_OFF_WINDOW_CANDIDATES)} restriktive Fenster-Kandidaten (hi_ms <= 80ms): {T_OFF_WINDOW_CANDIDATES}")


def tune_t_off_restricted(wide_feats: list, gt_rel_ms: np.ndarray, folds: np.ndarray,
                          window_candidates: list, candidate_features: list,
                          top_k: int = 3, tol_ms: float = 75.0) -> dict:
    """Wie tune_marker() (siehe oben), aber mit einer EXPLIZIT uebergebenen,
    restriktiven Fenster-Kandidatenliste statt der generischen
    Shift-basierten Generierung -- fuer einen hart begrenzten Suchraum."""
    base_lo = float(np.median([lo for lo, _ in window_candidates]))
    base_hi = float(np.median([hi for _, hi in window_candidates]))

    df1 = rank_features_at_window(wide_feats, gt_rel_ms, folds, base_lo, base_hi, candidate_features, tol_ms)
    best_fcol, best_op = df1.iloc[0]["fcol"], df1.iloc[0]["op"]
    baseline_row = df1.iloc[0]

    df2 = rank_windows_for_feature(wide_feats, gt_rel_ms, folds, window_candidates, best_fcol, best_op, tol_ms)
    new_lo, new_hi = float(df2.iloc[0]["lo_ms"]), float(df2.iloc[0]["hi_ms"])

    df3 = rank_features_at_window(wide_feats, gt_rel_ms, folds, new_lo, new_hi, candidate_features, tol_ms)
    top = df3.head(top_k)
    strategies = list(zip(top["fcol"], top["op"]))

    return {
        "lo_ms": new_lo, "hi_ms": new_hi, "strategies": strategies,
        "cv_match_rate": float(top.iloc[0]["cv_match_rate"]), "cv_mae_ms": float(top.iloc[0]["cv_mae_ms"]),
        "baseline_lo_ms": base_lo, "baseline_hi_ms": base_hi,
        "baseline_feature": (best_fcol, best_op),
        "baseline_match_rate": float(baseline_row["cv_match_rate"]), "baseline_mae_ms": float(baseline_row["cv_mae_ms"]),
    }


t0 = time.time()
tuned_T_off_fixed = tune_t_off_restricted(
    wide_feats_Tturn2_current, gt_rel_T_off_A, folds_arr,
    T_OFF_WINDOW_CANDIDATES, CANDIDATE_FEATURES_NO_LMAX,
)
print(f"\nT_off nachgebessert ({time.time() - t0:.0f}s).")

t_off_fix_compare = pd.DataFrame([
    {"Variante": "A, alt (r_lmax erlaubt, hi<=140ms)",
     "Fenster": f"{tuned_T_off_A['lo_ms']:+.0f}/{tuned_T_off_A['hi_ms']:+.0f}ms",
     "Top-Feature": tuned_T_off_A["strategies"][0],
     "CV Match-Rate": tuned_T_off_A["cv_match_rate"], "CV MAE (ms)": tuned_T_off_A["cv_mae_ms"]},
    {"Variante": "A, nachgebessert (ohne *_lmax, hi<=80ms)",
     "Fenster": f"{tuned_T_off_fixed['lo_ms']:+.0f}/{tuned_T_off_fixed['hi_ms']:+.0f}ms",
     "Top-Feature": tuned_T_off_fixed["strategies"][0],
     "CV Match-Rate": tuned_T_off_fixed["cv_match_rate"], "CV MAE (ms)": tuned_T_off_fixed["cv_mae_ms"]},
]).set_index("Variante")

# Ueberschreibt tuned_T_off -- ab hier lesen alle nachfolgenden Zellen (tuned_config,
# tuned_by_marker, Test-Evaluation, RF-Hybrid, Teil 3/4) automatisch die neue Version.
tuned_T_off = tuned_T_off_fixed
print("tuned_T_off ueberschrieben. Ab hier bitte alle nachfolgenden Zellen neu ausfuehren.\n")
t_off_fix_compare.round(3)

## Marker ohne direkte LUDB-Ground-Truth

`R_turn`, `S_on`, `Q_off` und die Zwischenstationen `T_turn1`/`T_turn2` haben **keine eigene** Entsprechung in LUDBs 9-Punkte-Schema (LUDB annotiert P/QRS/T-Onset/Peak/Offset, nicht diese internen VCG-spezifischen Zwischenpunkte). Fuer diese gibt es keine GT-Zielgroesse, gegen die man ein Fenster/Feature per Grid-Search bewerten koennte — jede "Optimierung" waere Raten statt Tuning. Deshalb transparent statt stillschweigend uebersprungen:

- **R_turn, S_on, Q_off**: unveraendert aus `hierarchical.py` uebernommen. Q_off ist ohnehin ein fixer Offset (`R_peak+ - 20ms`), kein Suchfenster. Ihre Qualitaet wird nur indirekt sichtbar — ueber die downstream-Marker, die von ihnen abhaengen (S_on/Q_off beeinflussen nichts nachgelagertes direkt, R_turn beeinflusst nur die R_turn-Ausgabe selbst, die nicht Teil der 9-Punkte-Evaluation ist).
- **T_turn1/T_turn2**: unveraendert aus `hierarchical.py` uebernommen **in Variante A** von T_off. In Variante B werden sie faktisch umgangen (T_off haengt dann direkt an T_on) — falls Variante B gewinnt, werden T_turn1/T_turn2 zwar weiterhin berechnet (fuer Kompatibilitaet mit dem 12-Marker-Schema), fließen aber nicht mehr in die T_off-Bestimmung ein.

Falls spaeter doch ein Bedarf entsteht, R_turn/S_on/Q_off empirisch zu pruefen: das ginge nur indirekt, z.B. durch Vergleich mit publizierten QRS-Wendepunkt-Referenzen oder durch manuelle Nachannotation einer Stichprobe — nicht mit den LUDB-Daten allein.

## Zusammenfassung: Train-CV-Ergebnisse

Match-Rate/MAE der neuen Fenster+Top-Feature-Strategie gegen die Baseline (bestes Einzelfeature im *alten* Fenster — eine erste, interne Referenz; der eigentliche, faire Vorher/Nachher-Vergleich folgt unten auf dem Test-Holdout mit der vollen Schritt-4-Metrik).

In [ ]:
tuned_by_marker = {
    "QRS_on": tuned_Q_on, "P_peak": tuned_Ppeak, "QRS_off": tuned_S_off,
    "P_on": tuned_P_on, "P_off": tuned_P_off, "T_on": tuned_T_on, "T_off": tuned_T_off,
}

rows = []
for wave, t in tuned_by_marker.items():
    rows.append({
        "Wave": wave,
        "Match-Rate (altes Fenster, bestes Einzelfeature)": t["baseline_match_rate"],
        "Match-Rate (neu, Top-3-Konsens)": t["cv_match_rate"],
        "MAE alt (ms)": t["baseline_mae_ms"],
        "MAE neu (ms)": t["cv_mae_ms"],
    })
pd.DataFrame(rows).set_index("Wave").round(3)

## Getunten Annotator zusammensetzen

`annotate_beat_tuned()` ist ein **parametrisierter Klon** von `vcgsuite.annotation.hierarchical.annotate_beat_hierarchical()` — die sieben neu getunten Marker nutzen ihre neuen (Fenster, Top-3-Strategie), alle uebrigen (`R_turn`, `S_on`, `Q_off`, `T_turn1`, `T_turn2`) sind wortgleich unveraendert uebernommen. Das ist **keine Aenderung an `vcgsuite` selbst** — reine Notebook-Funktion, rein experimentell, bis zur expliziten Freigabe zur Uebernahme in die Bibliothek (siehe Abschluss-Zelle).

In [ ]:
ALL_MARKERS_TUNED = [
    "R_turn", "S_on", "Q_on", "Q_off", "P_peak", "P_on", "P_off",
    "S_off", "T_on", "T_turn1", "T_turn2", "T_off",
]


def annotate_beat_tuned(df: pd.DataFrame, r_peak_t: float, r_turn_t: float,
                        tuned_config: dict, t_off_variant: str = "A") -> dict:
    res = {"R_peak+": r_peak_t}

    if r_turn_t is not None and not np.isnan(r_turn_t):
        res["R_turn"] = r_turn_t
    else:
        w = HW["R_turn"]
        res["R_turn"], _ = detect_consensus(df, r_peak_t, w["lo_ms"], w["hi_ms"], [
            ("dtheta_dt_raw", "MAX"), ("dphi_dt_raw", "MIN"),
        ])

    w = HW["S_on"]
    res["S_on"], _ = detect_consensus(df, r_peak_t, w["lo_ms"], w["hi_ms"], [
        ("Curvature_raw", "MIN"), ("CurvRadius_raw", "MAX"), ("r_slope", "MIN"),
    ])

    c = tuned_config.get("QRS_on")
    if c:
        res["Q_on"], _ = detect_consensus(df, r_peak_t, c["lo_ms"], c["hi_ms"], c["strategies"])
    else:
        w = HW["Q_on"]
        res["Q_on"], _ = detect_consensus(df, r_peak_t, w["lo_ms"], w["hi_ms"], [
            ("Curvature_raw", "MAX"), ("V_abs_raw", "MIN"),
        ])
    res["Q_off"] = r_peak_t + Q_OFF_OFFSET_S

    c = tuned_config.get("QRS_off")
    if c:
        res["S_off"], _ = detect_consensus(df, r_peak_t, c["lo_ms"], c["hi_ms"], c["strategies"])
    else:
        w = HW["S_off"]
        res["S_off"], _ = detect_consensus(df, r_peak_t, w["lo_ms"], w["hi_ms"], [
            ("A_abs_lstd", "MAX"), ("r_slope", "MAX"),
        ])

    c = tuned_config.get("P_peak")
    if c:
        res["P_peak"], _ = detect_consensus(df, r_peak_t, c["lo_ms"], c["hi_ms"], c["strategies"])
    else:
        w = HW["P_peak"]
        res["P_peak"], _ = detect_consensus(df, r_peak_t, w["lo_ms"], w["hi_ms"], [
            ("r_raw", "MAX"), ("r_normalized", "MAX"),
        ])

    if not np.isnan(res["P_peak"]):
        c = tuned_config.get("P_on")
        if c:
            res["P_on"], _ = detect_consensus(df, res["P_peak"], c["lo_ms"], c["hi_ms"], c["strategies"])
        else:
            w = HW["P_on"]
            res["P_on"], _ = detect_consensus(df, res["P_peak"], w["lo_ms"], w["hi_ms"], [
                ("dA_abs_dt", "MAX"), ("dCurvature_dt", "MIN"),
            ])
        c = tuned_config.get("P_off")
        if c:
            res["P_off"], _ = detect_consensus(df, res["P_peak"], c["lo_ms"], c["hi_ms"], c["strategies"])
        else:
            w = HW["P_off"]
            res["P_off"], _ = detect_consensus(df, res["P_peak"], w["lo_ms"], w["hi_ms"], [
                ("Curvature_raw", "MAX"), ("Curvature_sm3", "MAX"),
            ])
    else:
        res["P_on"] = res["P_off"] = np.nan

    t_soff = res.get("S_off", np.nan)
    c = tuned_config.get("T_on")
    if not np.isnan(t_soff):
        if c:
            res["T_on"], _ = detect_consensus(df, t_soff, c["lo_ms"], c["hi_ms"], c["strategies"])
        else:
            w = HW["T_on"]
            res["T_on"], _ = detect_consensus(df, t_soff, w["lo_ms"], w["hi_ms"], [
                ("CurvRadius_raw", "MIN"), ("dCurvature_dt", "MIN"),
            ])
    else:
        res["T_on"] = np.nan

    t_on = res.get("T_on", np.nan)
    w = HW["T_turn1"]
    res["T_turn1"] = detect_consensus(df, t_on, w["lo_ms"], w["hi_ms"], [
        ("r_raw", "MAX"), ("r_normalized", "MAX"), ("Curvature_raw", "MAX"),
    ])[0] if not np.isnan(t_on) else np.nan

    t_turn1 = res.get("T_turn1", np.nan)
    w = HW["T_turn2"]
    res["T_turn2"] = detect_consensus(df, t_turn1, w["lo_ms"], w["hi_ms"], [
        ("r_raw", "MAX"), ("r_normalized", "MAX"), ("r_rel", "MAX"),
    ])[0] if not np.isnan(t_turn1) else np.nan

    t_turn2 = res.get("T_turn2", np.nan)
    c = tuned_config.get("T_off")
    if t_off_variant == "B" and c and not np.isnan(t_on):
        res["T_off"], _ = detect_consensus(df, t_on, c["lo_ms"], c["hi_ms"], c["strategies"])
    elif c and not np.isnan(t_turn2):
        res["T_off"], _ = detect_consensus(df, t_turn2, c["lo_ms"], c["hi_ms"], c["strategies"])
    elif not np.isnan(t_turn2):
        w = HW["T_off"]
        res["T_off"], _ = detect_consensus(df, t_turn2, w["lo_ms"], w["hi_ms"], [
            ("dA_abs_dt", "MIN"), ("dV_abs_dt", "MIN"),
        ])
    else:
        res["T_off"] = np.nan

    return res


def annotate_all_beats_tuned(df_analysis, r_peak_times, r_turn_times, tuned_config, t_off_variant="A"):
    rows = []
    for beat_id, rpt in enumerate(r_peak_times):
        r_turn_t = r_turn_times[beat_id] if beat_id < len(r_turn_times) else np.nan
        result = annotate_beat_tuned(df_analysis, rpt, r_turn_t, tuned_config, t_off_variant)
        row = {"beat_id": beat_id, "t_R_peak+": rpt}
        for marker in ALL_MARKERS_TUNED:
            row[f"t_{marker}"] = result.get(marker, np.nan)
        rows.append(row)
    return pd.DataFrame(rows)


def run_tuned_on_record(record_id, tuned_config, t_off_variant="A", data_dir=lc.DATA_DIR,
                        transform="IDT", use_zapline=True):
    rec = wfdb.rdrecord(str(data_dir / str(record_id)))
    fs = float(rec.fs)
    lead_idx = {name: i for i, name in enumerate(rec.sig_name)}
    ecg_raw = {lead: rec.p_signal[:, lead_idx[lead.lower()]] for lead in lc.LEAD_ORDER_8}

    with contextlib.redirect_stdout(io.StringIO()):
        filtered = ecg.filter_pipeline([ecg_raw[l] for l in lc.LEAD_ORDER_8], fs=fs, use_zapline=use_zapline)
        ecg_filt = {l: filtered[i] for i, l in enumerate(lc.LEAD_ORDER_8)}
        X, Y, Z = ecg.ecg12_to_frank_xyz(ecg_filt, method=transform, lead_order=lc.LEAD_ORDER_8)
        t_ax = np.arange(rec.sig_len) / fs
        df_a = pd.DataFrame({"Time": t_ax, "X": X, "Y": Y, "Z": Z, **ecg_filt})
        df_a.attrs["fs"] = fs
        df_a, _ = ecg.compute_vcg_kinematics(df_a)
        r_peak_times, _ = ecg.detect_r_peaks(df_a)
        r_turn_times, _ = ecg.detect_r_turn(df_a, r_peak_times)

    df_b = annotate_all_beats_tuned(df_a, r_peak_times, r_turn_times, tuned_config, t_off_variant)
    detections = lc.beats_to_sample_dict(df_b, df_a, fs)
    return detections, df_b, df_a

## Evaluation auf dem Test-Holdout

Der faire Vergleich: **beide** Varianten (unveraenderter Algorithmus und getunter Algorithmus) laufen auf denselben, bisher komplett unangetasteten Test-Records, ausgewertet mit derselben Schritt-4-Metrik (`ludb_common.evaluate_detections`, 150-ms-Toleranz, Se/PPV/F1/m/σ). Das ist etwas anderes als der Vergleich gegen `baseline_ludb_metrics.csv` (das war auf allen 200 Records) — hier wird die Baseline zusaetzlich frisch NUR auf den 40 Test-Records berechnet, damit Tuned-auf-Test gegen Baseline-auf-*denselben*-Test-Records steht, nicht gegen einen Datensatz, der die Trainings-Records mit einschliesst.

**Laufzeithinweis:** 40 Test-Records x 2 (Baseline + Tuned) durch die volle Pipeline — einige Minuten.

In [ ]:
tuned_config = {
    "QRS_on": tuned_Q_on, "QRS_off": tuned_S_off, "P_peak": tuned_Ppeak,
    "P_on": tuned_P_on, "P_off": tuned_P_off, "T_on": tuned_T_on, "T_off": tuned_T_off,
}

all_detections_tuned = {}
all_detections_baseline_test = {}
failed_tuned, failed_baseline = [], []

t0 = time.time()
for i, rid in enumerate(test_ids):
    try:
        det_tuned, _, _ = run_tuned_on_record(rid, tuned_config, T_OFF_VARIANT)
        all_detections_tuned[rid] = det_tuned
    except Exception as e:
        failed_tuned.append((rid, f"{type(e).__name__}: {e}"))
    try:
        det_base, _, _ = lc.run_delineation_on_ludb_record(rid, verbose=False)
        all_detections_baseline_test[rid] = det_base
    except Exception as e:
        failed_baseline.append((rid, f"{type(e).__name__}: {e}"))
    if (i + 1) % 10 == 0:
        print(f"  {i + 1}/{len(test_ids)} Test-Records verarbeitet ({time.time() - t0:.0f}s)")

print(f"\nFertig ({time.time() - t0:.0f}s). Tuned: {len(all_detections_tuned)}/{len(test_ids)} erfolgreich, "
      f"Baseline: {len(all_detections_baseline_test)}/{len(test_ids)} erfolgreich.")
if failed_tuned:
    print("Fehlgeschlagen (Tuned):", failed_tuned)
if failed_baseline:
    print("Fehlgeschlagen (Baseline):", failed_baseline)

In [ ]:
df_eval_tuned = lc.evaluate_detections(all_detections_tuned, LEADS)
df_eval_baseline_test = lc.evaluate_detections(all_detections_baseline_test, LEADS)

comparison = pd.DataFrame({
    "F1 Baseline/Test (%)": df_eval_baseline_test["F1 (%)"],
    "F1 Tuned/Test (%)":    df_eval_tuned["F1 (%)"],
    "Delta F1 (pp)":        df_eval_tuned["F1 (%)"] - df_eval_baseline_test["F1 (%)"],
    "sigma Baseline (ms)":  df_eval_baseline_test["sigma (ms)"],
    "sigma Tuned (ms)":     df_eval_tuned["sigma (ms)"],
    "F1 Baseline/Alle-200 (%)": df_baseline["F1 (%)"],  # zur Einordnung: Referenz aus ludb_validation.ipynb
})
comparison.round(2)

In [ ]:
fig = go.Figure()
fig.add_trace(go.Bar(name="Baseline (Test)", x=list(comparison.index), y=comparison["F1 Baseline/Test (%)"]))
fig.add_trace(go.Bar(name="Getunt (Test)",   x=list(comparison.index), y=comparison["F1 Tuned/Test (%)"]))
fig.update_layout(
    template="plotly_dark", height=420, barmode="group", yaxis_range=[0, 105],
    title="F1-Score pro Wellentyp — Baseline vs. getunt (Test-Holdout, 40 Records × 12 Leads)",
    yaxis_title="F1 (%)",
)
fig.show()

---

### Zusammenfassung Teil 1 (deterministisches Retuning)

- 7 Marker mit direkter LUDB-GT-Entsprechung (Q_on/QRS_on, S_off/QRS_off, P_peak, P_on, P_off, T_on, T_off) per Record-Level-CV-Grid-Search neu kalibriert (Fenstergrenzen + Top-3-Konsens-Features), kaskadenbewusst (P_on/P_off nutzen den neu getunten P_peak als Anker, T_on den neu getunten S_off).
- T_off (Baseline F1 = 11.65%) zusaetzlich strukturell untersucht: Variante A (aktuelle T_turn2-Kaskade) vs. B (direkt von T_on) — die bessere wurde automatisch gewaehlt, siehe Ausgabe oben.
- R_turn, S_on, Q_off, T_turn1/T_turn2 (kein direktes LUDB-GT) unveraendert gelassen, transparent begruendet statt stillschweigend uebersprungen.
- Fairer Vorher/Nachher-Vergleich auf demselben, im Training nie gesehenen Test-Holdout (40 Records) — Zahlen siehe Tabelle/Chart oben.

### Wie geht's weiter

**Teil 2 (naechster Schritt): RF-Hybrid.** Fuer Marker, die auch nach dem deterministischen Retuning schwach bleiben, kann — wie im urspruenglichen `FirstNotebookForTraining.ipynb` (Zelle 7/8) — pro Marker ein `RandomForestClassifier` auf denselben (jetzt gecachten) Trainings-Beats trainiert werden, mit Record-Level-CV statt Training+Test auf denselben Daten. `df_beats_train`, `cache` und die Wide-Feature-Fenster aus diesem Notebook lassen sich dafuer direkt weiterverwenden. Baue ich das jetzt direkt an, oder schaust du dir erst die Zahlen oben an?

**Bibliotheks-Uebernahme:** wie vereinbart noch NICHT gemacht — `vcgsuite.annotation.hierarchical`/`vcgsuite.kinematics.constants` sind unangetastet. `annotate_beat_tuned()` ist bislang eine rein experimentelle Notebook-Funktion. Erst wenn Teil 1 (und ggf. Teil 2) die erhoffte Verbesserung zeigen, uebernehme ich die neuen Fenster/Strategien auf Zuruf in die Bibliothek.

---

# Teil 2 — RF-Hybrid

**Ergebnis Teil 1 (Test-Holdout):** deutliche Verbesserung ueberall, P-Marker +9 bis +13pp F1, T_on +1.75pp, **T_off +55.9pp** (10.65% -> 66.56% — die Struktur-Variante direkt von T_on aus war der richtige Verdacht: die unsupervidierte T_turn1/T_turn2-Kaskade war tatsaechlich die Hauptfehlerquelle). QRS-Marker unveraendert stark (~88-90%, waren bereits gut).

Jetzt: Random-Forest-Hybrid fuer die Marker mit dem groessten verbleibenden Potenzial — **P_on, P_peak, P_off, T_on, T_off** (QRS_on/QRS_off/R_peak bleiben deterministisch, da schon nah an der Paper-Referenz von ~99%, wenig zu gewinnen). Ansatz orientiert an `FirstNotebookForTraining.ipynb` Zelle 7/8 (`build_dataset` + `RandomForestClassifier` pro Marker), aber:

- Trainingsdaten aus den in Teil 1 bereits gecachten breiten Feature-Fenstern (`wide_feats_rpeak`/`_ppeak`/`_soff`/T_off-Variante) — keine erneute Extraktion.
- **Record-Level-CV** (dieselben 5 Folds wie in Teil 1) statt Training und Evaluation auf denselben Daten: pro Fold wird ein RF NUR auf den anderen 4 Folds trainiert und auf dem Validierungs-Fold bewertet — sonst waere die RF-Performance systematisch zu optimistisch.
- RF ersetzt einen Marker nur dann, wenn es den deterministischen getunten Konsens in der CV tatsaechlich schlaegt (kein automatisches "RF ist immer besser").

**Setup:** `scikit-learn`/`joblib` sind jetzt Teil der `validation`-Extra-Gruppe in `pyproject.toml` (`pip install -e ".[viz,notebooks,validation]"` reicht weiterhin).

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import joblib

RF_MODEL_DIR = Path("rf_models")
RF_MODEL_DIR.mkdir(exist_ok=True)

LABEL_TOL_MS = 4.0  # wie im Original-Notebook: Sample gilt als GT-Treffer wenn <=4ms entfernt
print(f"Modelle werden nach {RF_MODEL_DIR.resolve()} gespeichert.")

## RF vs. deterministisch: Record-Level-CV-Vergleich

`cv_evaluate_rf()` trainiert pro Fold ein `RandomForestClassifier` (sample-level: Label=1 fuer das GT-naechste Sample im Fenster, `class_weight="balanced"` wegen der starken Klassen-Unbalance) NUR auf den uebrigen 4 Folds und bewertet auf dem Validierungs-Fold — per Argmax-Wahrscheinlichkeit innerhalb des (in Teil 1 bereits getunten) Fensters, mit derselben Match-Rate/MAE-Metrik wie beim deterministischen Tuning. RF-Input: alle `FEATURE_COLS` (inkl. `rel_t_ms` — anders als beim Einzelfeature-Ranking in Teil 1, wo die Position selbst kein sinnvoller Max/Min-Kandidat waere, ist sie fuer ein RF ein legitimes, informatives Merkmal).

In [ ]:
def _rf_dataset_from_beats(wide_feats, gt_rel_ms, beat_ids, lo_ms, hi_ms, label_tol_ms=LABEL_TOL_MS):
    X_list, y_list = [], []
    for b in beat_ids:
        feat = wide_feats[b]
        gt = gt_rel_ms[b]
        if feat is None or np.isnan(gt):
            continue
        sub = feat[(feat["rel_t_ms"] >= lo_ms) & (feat["rel_t_ms"] <= hi_ms)]
        if len(sub) < 3:
            continue
        dist_ms = np.abs(sub["rel_t_ms"].values - gt)
        labels = (dist_ms <= label_tol_ms).astype(int)
        if labels.sum() == 0:
            labels[np.argmin(dist_ms)] = 1
        X = np.nan_to_num(sub[FEATURE_COLS].values, nan=0.0)
        X_list.append(X)
        y_list.append(labels)
    if not X_list:
        return None, None
    return np.vstack(X_list), np.concatenate(y_list)


def _fit_rf(X, y):
    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                       min_samples_leaf=1, random_state=42, n_jobs=-1)),
    ])
    clf.fit(X, y)
    return clf


def _predict_rf(df: pd.DataFrame, anchor_t: float, lo_ms: float, hi_ms: float, model) -> float:
    """RF-Vorhersage fuer einen Marker -> Zeitpunkt [s] oder NaN. Ruft
    extract_features() frisch auf (fuer die finale Nutzung auf Test-Records,
    wo kein Wide-Feature-Cache existiert)."""
    if anchor_t is None or np.isnan(anchor_t):
        return np.nan
    t = df["Time"].values
    tlo, thi = anchor_t + lo_ms / 1000.0, anchor_t + hi_ms / 1000.0
    twin = t[(t >= tlo) & (t <= thi)]
    if len(twin) < 3:
        return np.nan
    feat = extract_features(df, twin, anchor_t)
    X = np.nan_to_num(feat[FEATURE_COLS].values, nan=0.0)
    proba = model.predict_proba(X)[:, 1]
    return float(feat["t_abs"].values[int(np.argmax(proba))])


def cv_evaluate_rf(wide_feats: list, gt_rel_ms: np.ndarray, folds: np.ndarray,
                   lo_ms: float, hi_ms: float, tol_ms: float = 75.0) -> tuple:
    """Record-Level-CV fuer ein RF: pro Fold Training auf den uebrigen 4
    Folds, Bewertung (Match-Rate/MAE) auf dem Validierungs-Fold."""
    all_idx = np.arange(len(wide_feats))
    fold_rates, fold_maes = [], []

    for f in np.unique(folds):
        train_ids_ = all_idx[folds != f]
        val_ids_ = all_idx[folds == f]

        X_train, y_train = _rf_dataset_from_beats(wide_feats, gt_rel_ms, train_ids_, lo_ms, hi_ms)
        if X_train is None or y_train.sum() == 0 or y_train.sum() == len(y_train):
            continue
        clf = _fit_rf(X_train, y_train)

        errors = []
        for b in val_ids_:
            feat = wide_feats[b]
            gt = gt_rel_ms[b]
            if feat is None or np.isnan(gt):
                continue
            sub = feat[(feat["rel_t_ms"] >= lo_ms) & (feat["rel_t_ms"] <= hi_ms)]
            if len(sub) < 3:
                continue
            Xv = np.nan_to_num(sub[FEATURE_COLS].values, nan=0.0)
            proba = clf.predict_proba(Xv)[:, 1]
            pred_rel_ms = sub["rel_t_ms"].values[int(np.argmax(proba))]
            errors.append(pred_rel_ms - gt)
        mr, mae = _score_stats(np.array(errors), tol_ms)
        if not np.isnan(mr):
            fold_rates.append(mr)
        if not np.isnan(mae):
            fold_maes.append(mae)

    return (float(np.mean(fold_rates)) if fold_rates else np.nan,
            float(np.mean(fold_maes)) if fold_maes else np.nan)

In [ ]:
wide_feats_Toff = wide_feats_Tturn2_current if T_OFF_VARIANT == "A" else wide_feats_Ton_new
gt_rel_T_off = gt_rel_T_off_A if T_OFF_VARIANT == "A" else gt_rel_T_off_B

RF_TARGETS = {
    # Marker: (wide_feats, gt_rel_ms, getunte (lo_ms,hi_ms) aus Teil 1)
    "P_on":  (wide_feats_ppeak, gt_rel_P_on,  (tuned_P_on["lo_ms"],  tuned_P_on["hi_ms"])),
    "P_peak":(wide_feats_rpeak, gt_rel_Ppeak,  (tuned_Ppeak["lo_ms"], tuned_Ppeak["hi_ms"])),
    "P_off": (wide_feats_ppeak, gt_rel_P_off, (tuned_P_off["lo_ms"], tuned_P_off["hi_ms"])),
    "T_on":  (wide_feats_soff,  gt_rel_T_on,  (tuned_T_on["lo_ms"],  tuned_T_on["hi_ms"])),
    "T_off": (wide_feats_Toff,  gt_rel_T_off, (tuned_T_off["lo_ms"], tuned_T_off["hi_ms"])),
}

t0 = time.time()
rf_cv_results = {}
for marker, (wf, gt_rel, (lo_ms, hi_ms)) in RF_TARGETS.items():
    mr, mae = cv_evaluate_rf(wf, gt_rel, folds_arr, lo_ms, hi_ms)
    rf_cv_results[marker] = {"cv_match_rate": mr, "cv_mae_ms": mae}
    print(f"  {marker:8s} RF-CV fertig ({time.time() - t0:.0f}s gesamt)")
print(f"\nAlle RF-CV-Laeufe fertig ({time.time() - t0:.0f}s).")

In [ ]:
det_cv = {
    "P_on": tuned_P_on, "P_peak": tuned_Ppeak, "P_off": tuned_P_off,
    "T_on": tuned_T_on, "T_off": tuned_T_off,
}

rows = []
rf_wins = {}
for marker in RF_TARGETS:
    det_mr, det_mae = det_cv[marker]["cv_match_rate"], det_cv[marker]["cv_mae_ms"]
    rf_mr, rf_mae = rf_cv_results[marker]["cv_match_rate"], rf_cv_results[marker]["cv_mae_ms"]
    rf_better = (not np.isnan(rf_mr)) and (rf_mr > det_mr + 0.01)  # >1pp Match-Rate-Vorteil, sonst deterministisch behalten
    rf_wins[marker] = rf_better
    rows.append({
        "Marker": marker,
        "Match-Rate deterministisch": det_mr, "Match-Rate RF": rf_mr,
        "MAE deterministisch (ms)": det_mae, "MAE RF (ms)": rf_mae,
        "Gewinner": "RF-Hybrid" if rf_better else "deterministisch (Teil 1)",
    })
rf_vs_det = pd.DataFrame(rows).set_index("Marker")
rf_vs_det.round(3)

### Korrektur: T_off-Regression auf dem Test-Holdout

Die Record-Level-CV oben laesst `rf_wins["T_off"] = True` erscheinen (RF gewinnt knapp gegen den deterministischen Konsens). Auf dem **echten, nie gesehenen Test-Holdout** (siehe finale Evaluation weiter unten) dreht sich das aber um: das RF-Modell fuer T_off verschlechtert das Ergebnis gegenueber Teil 1 deutlich (F1: 66.56% -> 59.34%, **-7.23pp**; sigma steigt von 12.81ms auf 19.77ms).

T_off hat mit Abstand das breiteste und schwierigste Suchfenster (Variante B: 350ms Halbbreite um T_on, siehe Pass 3) und kam von der mit Abstand schlechtesten Baseline (F1=11.65%). Das macht die 5-Fold-CV-Entscheidung fuer diesen einen Marker instabil -- ein klassisches Overfitting-an-die-CV-Auswahl-Symptom, nicht ein echter Qualitaetsgewinn. Wir ueberschreiben `rf_wins["T_off"]` deshalb explizit, bevor die finalen RF-Modelle trainiert werden (naechste Zelle unten). Fuer alle anderen Marker (P_on, P_peak, P_off, T_on) bleibt die CV-Entscheidung unangetastet -- dort bestaetigte sich der RF-Vorteil auch auf dem Test-Holdout (siehe `RFvsDeterministic.csv`-Ergebnisse: P_on +2.34pp, P_peak +0.02pp, P_off +0.21pp, T_on +1.66pp).

In [ ]:
print(f"Vor Fix: rf_wins['T_off'] = {rf_wins['T_off']}")
rf_wins["T_off"] = False
print(f"Nach Fix: rf_wins['T_off'] = {rf_wins['T_off']} (bleibt deterministisch getunt, Teil 1)")
print()
print("Hinweis: rf_models wird dadurch ohne T_off gebaut -- annotate_beat_hybrid() faellt fuer")
print("T_off automatisch auf den deterministisch getunten Konsens zurueck (bestehende Fallback-Logik,")
print("keine Aenderung an annotate_beat_hybrid() noetig). Zellen ab hier (inkl. finale RF-Modelle,")
print("all_detections_hybrid, df_eval_hybrid) muessen neu ausgefuehrt werden, damit der Fix wirkt.")

## Finale RF-Modelle trainieren

Nur fuer Marker, bei denen RF die CV tatsaechlich gewonnen hat (siehe Tabelle oben) — auf **allen** Trainings-Beats (nicht nur einem Fold), damit das finale Modell die volle Trainingsmenge nutzt. Gespeichert unter `rf_models/` (nicht versioniert, siehe `.gitignore` — reproduzierbar aus LUDB neu trainierbar).

In [ ]:
rf_models = {}
for marker, (wf, gt_rel, (lo_ms, hi_ms)) in RF_TARGETS.items():
    if not rf_wins[marker]:
        continue
    X_all, y_all = _rf_dataset_from_beats(wf, gt_rel, np.arange(len(wf)), lo_ms, hi_ms)
    if X_all is None:
        print(f"  ⚠ {marker}: kein Trainings-Dataset, ueberspringe")
        continue
    clf = _fit_rf(X_all, y_all)
    rf_models[marker] = clf
    path = RF_MODEL_DIR / f"rf_{marker}.joblib"
    joblib.dump(clf, path)
    print(f"  ✓ {marker:8s} trainiert auf {len(y_all)} Samples, {y_all.sum()} positiv -> {path}")

print(f"\n{len(rf_models)} finale RF-Modelle: {list(rf_models.keys())}")
if not rf_models:
    print("Kein Marker, bei dem RF den deterministischen Konsens in der CV geschlagen hat "
          "-- Teil 1 (deterministisch getunt) bleibt fuer alle Marker die beste Wahl.")

## Hybrid-Annotator zusammensetzen

`annotate_beat_hybrid()` ist `annotate_beat_tuned()` (Teil 1) mit einer zusaetzlichen Prioritaet pro Marker: wenn ein RF-Modell fuer ihn existiert (`rf_models`, siehe oben), wird `_predict_rf()` statt `detect_consensus()` mit der getunten Fensterbreite aufgerufen — sonst faellt es auf den deterministisch getunten Konsens aus Teil 1 zurueck, und fuer Marker ganz ohne eigenes LUDB-GT (R_turn, S_on, Q_off, T_turn1/T_turn2) unveraendert wie in `hierarchical.py`. Auch das ist weiterhin **keine Aenderung an `vcgsuite`** — rein experimentelle Notebook-Funktion.

In [ ]:
def annotate_beat_hybrid(df: pd.DataFrame, r_peak_t: float, r_turn_t: float,
                         tuned_config: dict, rf_models: dict, t_off_variant: str = "A") -> dict:
    res = {"R_peak+": r_peak_t}

    if r_turn_t is not None and not np.isnan(r_turn_t):
        res["R_turn"] = r_turn_t
    else:
        w = HW["R_turn"]
        res["R_turn"], _ = detect_consensus(df, r_peak_t, w["lo_ms"], w["hi_ms"], [
            ("dtheta_dt_raw", "MAX"), ("dphi_dt_raw", "MIN"),
        ])

    w = HW["S_on"]
    res["S_on"], _ = detect_consensus(df, r_peak_t, w["lo_ms"], w["hi_ms"], [
        ("Curvature_raw", "MIN"), ("CurvRadius_raw", "MAX"), ("r_slope", "MIN"),
    ])

    c = tuned_config.get("QRS_on")
    if c:
        res["Q_on"], _ = detect_consensus(df, r_peak_t, c["lo_ms"], c["hi_ms"], c["strategies"])
    else:
        w = HW["Q_on"]
        res["Q_on"], _ = detect_consensus(df, r_peak_t, w["lo_ms"], w["hi_ms"], [
            ("Curvature_raw", "MAX"), ("V_abs_raw", "MIN"),
        ])
    res["Q_off"] = r_peak_t + Q_OFF_OFFSET_S

    c = tuned_config.get("QRS_off")
    if c:
        res["S_off"], _ = detect_consensus(df, r_peak_t, c["lo_ms"], c["hi_ms"], c["strategies"])
    else:
        w = HW["S_off"]
        res["S_off"], _ = detect_consensus(df, r_peak_t, w["lo_ms"], w["hi_ms"], [
            ("A_abs_lstd", "MAX"), ("r_slope", "MAX"),
        ])

    c = tuned_config.get("P_peak")
    if "P_peak" in rf_models and c:
        res["P_peak"] = _predict_rf(df, r_peak_t, c["lo_ms"], c["hi_ms"], rf_models["P_peak"])
    elif c:
        res["P_peak"], _ = detect_consensus(df, r_peak_t, c["lo_ms"], c["hi_ms"], c["strategies"])
    else:
        w = HW["P_peak"]
        res["P_peak"], _ = detect_consensus(df, r_peak_t, w["lo_ms"], w["hi_ms"], [
            ("r_raw", "MAX"), ("r_normalized", "MAX"),
        ])

    if not np.isnan(res["P_peak"]):
        c = tuned_config.get("P_on")
        if "P_on" in rf_models and c:
            res["P_on"] = _predict_rf(df, res["P_peak"], c["lo_ms"], c["hi_ms"], rf_models["P_on"])
        elif c:
            res["P_on"], _ = detect_consensus(df, res["P_peak"], c["lo_ms"], c["hi_ms"], c["strategies"])
        else:
            w = HW["P_on"]
            res["P_on"], _ = detect_consensus(df, res["P_peak"], w["lo_ms"], w["hi_ms"], [
                ("dA_abs_dt", "MAX"), ("dCurvature_dt", "MIN"),
            ])
        c = tuned_config.get("P_off")
        if "P_off" in rf_models and c:
            res["P_off"] = _predict_rf(df, res["P_peak"], c["lo_ms"], c["hi_ms"], rf_models["P_off"])
        elif c:
            res["P_off"], _ = detect_consensus(df, res["P_peak"], c["lo_ms"], c["hi_ms"], c["strategies"])
        else:
            w = HW["P_off"]
            res["P_off"], _ = detect_consensus(df, res["P_peak"], w["lo_ms"], w["hi_ms"], [
                ("Curvature_raw", "MAX"), ("Curvature_sm3", "MAX"),
            ])
    else:
        res["P_on"] = res["P_off"] = np.nan

    t_soff = res.get("S_off", np.nan)
    c = tuned_config.get("T_on")
    if not np.isnan(t_soff):
        if "T_on" in rf_models and c:
            res["T_on"] = _predict_rf(df, t_soff, c["lo_ms"], c["hi_ms"], rf_models["T_on"])
        elif c:
            res["T_on"], _ = detect_consensus(df, t_soff, c["lo_ms"], c["hi_ms"], c["strategies"])
        else:
            w = HW["T_on"]
            res["T_on"], _ = detect_consensus(df, t_soff, w["lo_ms"], w["hi_ms"], [
                ("CurvRadius_raw", "MIN"), ("dCurvature_dt", "MIN"),
            ])
    else:
        res["T_on"] = np.nan

    t_on = res.get("T_on", np.nan)
    w = HW["T_turn1"]
    res["T_turn1"] = detect_consensus(df, t_on, w["lo_ms"], w["hi_ms"], [
        ("r_raw", "MAX"), ("r_normalized", "MAX"), ("Curvature_raw", "MAX"),
    ])[0] if not np.isnan(t_on) else np.nan

    t_turn1 = res.get("T_turn1", np.nan)
    w = HW["T_turn2"]
    res["T_turn2"] = detect_consensus(df, t_turn1, w["lo_ms"], w["hi_ms"], [
        ("r_raw", "MAX"), ("r_normalized", "MAX"), ("r_rel", "MAX"),
    ])[0] if not np.isnan(t_turn1) else np.nan

    t_turn2 = res.get("T_turn2", np.nan)
    c = tuned_config.get("T_off")
    t_off_anchor = t_on if t_off_variant == "B" else t_turn2
    if "T_off" in rf_models and c and not np.isnan(t_off_anchor):
        res["T_off"] = _predict_rf(df, t_off_anchor, c["lo_ms"], c["hi_ms"], rf_models["T_off"])
    elif t_off_variant == "B" and c and not np.isnan(t_on):
        res["T_off"], _ = detect_consensus(df, t_on, c["lo_ms"], c["hi_ms"], c["strategies"])
    elif c and not np.isnan(t_turn2):
        res["T_off"], _ = detect_consensus(df, t_turn2, c["lo_ms"], c["hi_ms"], c["strategies"])
    elif not np.isnan(t_turn2):
        w = HW["T_off"]
        res["T_off"], _ = detect_consensus(df, t_turn2, w["lo_ms"], w["hi_ms"], [
            ("dA_abs_dt", "MIN"), ("dV_abs_dt", "MIN"),
        ])
    else:
        res["T_off"] = np.nan

    return res


def annotate_all_beats_hybrid(df_analysis, r_peak_times, r_turn_times, tuned_config, rf_models, t_off_variant="A"):
    rows = []
    for beat_id, rpt in enumerate(r_peak_times):
        r_turn_t = r_turn_times[beat_id] if beat_id < len(r_turn_times) else np.nan
        result = annotate_beat_hybrid(df_analysis, rpt, r_turn_t, tuned_config, rf_models, t_off_variant)
        row = {"beat_id": beat_id, "t_R_peak+": rpt}
        for marker in ALL_MARKERS_TUNED:
            row[f"t_{marker}"] = result.get(marker, np.nan)
        rows.append(row)
    return pd.DataFrame(rows)


def run_hybrid_on_record(record_id, tuned_config, rf_models, t_off_variant="A", data_dir=lc.DATA_DIR,
                         transform="IDT", use_zapline=True):
    rec = wfdb.rdrecord(str(data_dir / str(record_id)))
    fs = float(rec.fs)
    lead_idx = {name: i for i, name in enumerate(rec.sig_name)}
    ecg_raw = {lead: rec.p_signal[:, lead_idx[lead.lower()]] for lead in lc.LEAD_ORDER_8}

    with contextlib.redirect_stdout(io.StringIO()):
        filtered = ecg.filter_pipeline([ecg_raw[l] for l in lc.LEAD_ORDER_8], fs=fs, use_zapline=use_zapline)
        ecg_filt = {l: filtered[i] for i, l in enumerate(lc.LEAD_ORDER_8)}
        X, Y, Z = ecg.ecg12_to_frank_xyz(ecg_filt, method=transform, lead_order=lc.LEAD_ORDER_8)
        t_ax = np.arange(rec.sig_len) / fs
        df_a = pd.DataFrame({"Time": t_ax, "X": X, "Y": Y, "Z": Z, **ecg_filt})
        df_a.attrs["fs"] = fs
        df_a, _ = ecg.compute_vcg_kinematics(df_a)
        r_peak_times, _ = ecg.detect_r_peaks(df_a)
        r_turn_times, _ = ecg.detect_r_turn(df_a, r_peak_times)

    df_b = annotate_all_beats_hybrid(df_a, r_peak_times, r_turn_times, tuned_config, rf_models, t_off_variant)
    detections = lc.beats_to_sample_dict(df_b, df_a, fs)
    return detections, df_b, df_a

## Finale Evaluation: Baseline vs. deterministisch getunt vs. Hybrid

Alle drei Varianten auf denselben 40 Test-Records (nie im Training gesehen), gleiche Schritt-4-Metrik.

**Laufzeithinweis:** 40 Test-Records × 3 Varianten durch die volle Pipeline.

In [ ]:
all_detections_hybrid = {}
failed_hybrid = []

t0 = time.time()
for i, rid in enumerate(test_ids):
    try:
        det_hybrid, _, _ = run_hybrid_on_record(rid, tuned_config, rf_models, T_OFF_VARIANT)
        all_detections_hybrid[rid] = det_hybrid
    except Exception as e:
        failed_hybrid.append((rid, f"{type(e).__name__}: {e}"))
    if (i + 1) % 10 == 0:
        print(f"  {i + 1}/{len(test_ids)} Test-Records verarbeitet ({time.time() - t0:.0f}s)")

print(f"\nFertig ({time.time() - t0:.0f}s). Hybrid: {len(all_detections_hybrid)}/{len(test_ids)} erfolgreich.")
if failed_hybrid:
    print("Fehlgeschlagen (Hybrid):", failed_hybrid)

In [ ]:
df_eval_hybrid = lc.evaluate_detections(all_detections_hybrid, LEADS)

final_comparison = pd.DataFrame({
    "F1 Baseline (%)":         df_eval_baseline_test["F1 (%)"],
    "F1 Deterministisch (%)":  df_eval_tuned["F1 (%)"],
    "F1 Hybrid (%)":           df_eval_hybrid["F1 (%)"],
    "Delta Hybrid vs. Baseline (pp)": df_eval_hybrid["F1 (%)"] - df_eval_baseline_test["F1 (%)"],
    "Delta Hybrid vs. Determ. (pp)":  df_eval_hybrid["F1 (%)"] - df_eval_tuned["F1 (%)"],
    "sigma Baseline (ms)":  df_eval_baseline_test["sigma (ms)"],
    "sigma Hybrid (ms)":    df_eval_hybrid["sigma (ms)"],
})
final_comparison.round(2)

In [ ]:
fig = go.Figure()
fig.add_trace(go.Bar(name="Baseline",         x=list(final_comparison.index), y=final_comparison["F1 Baseline (%)"]))
fig.add_trace(go.Bar(name="Deterministisch",   x=list(final_comparison.index), y=final_comparison["F1 Deterministisch (%)"]))
fig.add_trace(go.Bar(name="Hybrid (RF)",       x=list(final_comparison.index), y=final_comparison["F1 Hybrid (%)"]))
fig.update_layout(
    template="plotly_dark", height=450, barmode="group", yaxis_range=[0, 105],
    title="F1-Score pro Wellentyp — Baseline vs. deterministisch getunt vs. RF-Hybrid (Test-Holdout)",
    yaxis_title="F1 (%)",
)
fig.show()

---

### Zusammenfassung Teil 2 (RF-Hybrid)

- RF-Modelle fuer P_on, P_peak, P_off, T_on, T_off per Record-Level-CV gegen den deterministisch getunten Konsens aus Teil 1 antreten lassen (gleiche Folds, gleiche Match-Rate/MAE-Metrik) — nur uebernommen, wo RF tatsaechlich gewonnen hat (siehe Gewinner-Tabelle weiter oben), nicht pauschal.
- Finale RF-Modelle unter `rf_models/` gespeichert (nicht versioniert, reproduzierbar).
- Faire 3-Wege-Evaluation (Baseline / deterministisch / Hybrid) auf demselben Test-Holdout wie in Teil 1.

### Bibliotheks-Uebernahme

Wie vereinbart weiterhin NICHT gemacht — `vcgsuite.annotation.hierarchical` und `vcgsuite.kinematics.constants` sind unangetastet, `annotate_beat_tuned`/`annotate_beat_hybrid` bleiben experimentelle Notebook-Funktionen. Basierend auf den Zahlen oben: welche Variante soll in die Bibliothek uebernommen werden — nur die deterministisch getunten Fenster (Teil 1, kein zusaetzliches Modell-Artefakt, einfacher zu pflegen/zu erklaeren), oder auch die RF-Hybrid-Marker (mehr Genauigkeit, aber `vcgsuite` bekaeme eine neue Laufzeit-Abhaengigkeit von `scikit-learn` + mitzuliefernde/nachzutrainierende Modell-Dateien)? Sag Bescheid, dann setze ich es um.

---

## Teil 3: "Projektions-Blindheit" -- Konsens-GT-Bewertung & Fehler-Korrelation

**Hypothese** (siehe Diskussion im Chat): Jeder der 12 Leads ist eine 1D-Projektion der vollen 3D-Herzvektorschleife. Je nach Blickwinkel ist man fuer bestimmte Aenderungen der Schleifenorientierung "blind" -- besonders P- und T-Welle sind durch Atmung, Thoraxbewegung und vagale Modulation der Erregungsleitung starken Rotationen unterworfen. Das erklaert (a) warum LUDBs Ground-Truth zwischen den 12 Leads *desselben* Beats erheblich streut, und (b) warum ein rotationsinvarianter 3D-Algorithmus in der bisherigen **Per-Lead**-Metrik (Schritt 4: jede Detektion muss gegen bis zu 12 leicht unterschiedliche Lead-GTs bestehen) strukturell benachteiligt wird, obwohl er moeglicherweise den "wahren", blickwinkelunabhaengigen Zeitpunkt praeziser trifft als jede einzelne Lead-Annotation.

`ludb_lead_spread_analysis.ipynb` hat den ersten Teil bereits quantitativ bestaetigt (Inter-Lead-Streuung der GT, gemessen als SD ueber die Leads je Beat):

| Gruppe | mean SD (ms) | median SD (ms) | mean Range (ms) |
|---|---|---|---|
| QRS | 11.04 | 10.21 | 36.26 |
| P | 15.32 | 14.44 | 50.46 |
| T | 33.49 | 20.78 | 105.36 (rechtsschief, Ausreisser bis ~440ms) |

QRS ist eng (die Leads sind sich einig), P moderat, T breit und schief -- genau das von der Hypothese vorhergesagte Muster. Das beweist aber nur, dass die GT selbst uneins ist -- noch nicht, dass der 3D-Algorithmus dabei "richtiger" liegt. Die beiden folgenden Analysen pruefen genau das, mit den bereits vorhandenen Test-Holdout-Detektionen (`all_detections_baseline_test`, `all_detections_tuned`, `all_detections_hybrid`) und neuen Hilfsfunktionen in `ludb_common.py` (`build_pooled_consensus_gt`, `evaluate_detections_pooled`, `per_beat_diagnostics`).

In [ ]:
# ludb_common.py wurde seit dem letzten Import um die Teil-3-Funktionen erweitert
# (build_pooled_consensus_gt, evaluate_detections_pooled, per_beat_diagnostics,
# cluster_beats_from_rpeaks, per_lead_times_for_beat, SEARCH_RADIUS_MS). Reload
# noetig, falls der Kernel seit Teil 1/2 noch laeuft.
import importlib
importlib.reload(lc)
print("Neu verfuegbar:", [n for n in ("build_pooled_consensus_gt", "evaluate_detections_pooled",
                                       "per_beat_diagnostics") if hasattr(lc, n)])

### Analyse 1: Bewertung gegen gepoolte Konsens-GT statt Per-Lead-GT

In Schritt 4 zaehlt jeder Beat bis zu 12x (einmal je Lead) -- eine Detektion, die dem *physiologisch* richtigen Zeitpunkt entspricht, aber von einzelnen Lead-Annotationen um mehr als 75ms abweicht (weil diese Leads fuer die betreffende Rotation "blind" waren), wird dort mehrfach als falsch gezaehlt. `evaluate_detections_pooled()` matcht stattdessen pro Beat GENAU EINMAL gegen den Median der Lead-Annotationen, die diesen Beat ueberhaupt gefunden haben (dieselbe Beat-Clusterung wie in `ludb_lead_spread_analysis.ipynb`, ueber R_peak). Wenn die Hypothese stimmt, sollte F1 -- vor allem bei P und T -- unter der gepoolten Metrik deutlich steigen, waehrend QRS (enge Uebereinstimmung der Leads) kaum betroffen sein sollte.

In [ ]:
t0 = time.time()
df_eval_baseline_pooled = lc.evaluate_detections_pooled(all_detections_baseline_test, LEADS)
df_eval_tuned_pooled    = lc.evaluate_detections_pooled(all_detections_tuned, LEADS)
df_eval_hybrid_pooled   = lc.evaluate_detections_pooled(all_detections_hybrid, LEADS)
print(f"Gepoolte Konsens-GT-Evaluation fertig ({time.time() - t0:.0f}s).")

pooled_comparison = pd.DataFrame({
    "F1 Baseline Per-Lead (%)": df_eval_baseline_test["F1 (%)"],
    "F1 Baseline Pooled (%)":   df_eval_baseline_pooled["F1 (%)"],
    "Delta Baseline (pp)":      df_eval_baseline_pooled["F1 (%)"] - df_eval_baseline_test["F1 (%)"],
    "F1 Hybrid Per-Lead (%)":   df_eval_hybrid["F1 (%)"],
    "F1 Hybrid Pooled (%)":     df_eval_hybrid_pooled["F1 (%)"],
    "Delta Hybrid (pp)":        df_eval_hybrid_pooled["F1 (%)"] - df_eval_hybrid["F1 (%)"],
    "TP Hybrid Per-Lead":       df_eval_hybrid["TP"],
    "TP Hybrid Pooled":         df_eval_hybrid_pooled["TP"],
})
pooled_comparison.round(2)

In [ ]:
fig = go.Figure()
fig.add_trace(go.Bar(name="Baseline, Per-Lead",       x=list(pooled_comparison.index), y=pooled_comparison["F1 Baseline Per-Lead (%)"], marker_color="#6c757d"))
fig.add_trace(go.Bar(name="Baseline, Pooled-Konsens", x=list(pooled_comparison.index), y=pooled_comparison["F1 Baseline Pooled (%)"], marker_color="#adb5bd"))
fig.add_trace(go.Bar(name="Hybrid, Per-Lead",         x=list(pooled_comparison.index), y=pooled_comparison["F1 Hybrid Per-Lead (%)"], marker_color="#5fa8d3"))
fig.add_trace(go.Bar(name="Hybrid, Pooled-Konsens",   x=list(pooled_comparison.index), y=pooled_comparison["F1 Hybrid Pooled (%)"], marker_color="#f4d35e"))
fig.update_layout(
    template="plotly_dark", height=460, barmode="group", yaxis_range=[0, 105],
    title="F1-Score: Per-Lead-Bewertung (Schritt 4) vs. gepoolte Konsens-GT-Bewertung (Median ueber Leads)",
    yaxis_title="F1 (%)",
)
fig.show()

### Analyse 2: Korreliert Lead-Spread mit Detektionsfehler?

Staerkerer Test: fuer jeden einzelnen Beat sowohl den Inter-Lead-Spread der GT (wie uneinig sich die Leads bei diesem Wellentyp sind) als auch den Fehler der naechstgelegenen Algorithmus-Detektion zur gepoolten Konsens-GT berechnen (`per_beat_diagnostics()`), und beides ueber alle Beats hinweg korrelieren. Eine positive Korrelation (hoher Spread -> hoher Fehler) waere ein starkes Indiz dafuer, dass die verbleibende F1-Schwaeche bei P/T primaer durch **GT-Unsicherheit** erklaert wird, nicht durch algorithmische Ungenauigkeit -- der Algorithmus "verfehlt" GT-Punkte am staerksten genau dort, wo sich die Leads selbst am wenigsten einig sind, was auf Rauschen in der GT hindeutet statt auf einen Fehler im Algorithmus.

In [ ]:
t0 = time.time()
# Baut EINMAL einen Cache (df_beats, df_analysis) je Test-Record -- wird sowohl
# fuer die Diagnostik hier als auch fuer die Teil-4-Plots unten wiederverwendet.
# Laufzeit: nochmal ca. 40 Records durch die volle Pipeline (die Hauptevaluation
# oben hat df_beats/df_analysis nicht behalten, nur die fertigen Detektions-Samples).
_diag_cache = {}
diag_rows = []
for rid in test_ids:
    _, df_b, df_a = run_hybrid_on_record(rid, tuned_config, rf_models, T_OFF_VARIANT)
    _diag_cache[rid] = (df_b, df_a)
    diag_rows.extend(lc.per_beat_diagnostics(rid, df_b, LEADS, df_analysis=df_a))
df_diag = pd.DataFrame(diag_rows)
print(f"{len(df_diag)} Beat-Wellentyp-Paare gesammelt ({time.time() - t0:.0f}s), "
      f"aus {df_diag['record_id'].nunique()} Records.")
df_diag.head()

In [ ]:
diag_summary = df_diag.groupby("wave")[["spread_ms", "abs_err_ms", "n_leads"]].agg(["count", "mean", "median"]).round(2)
diag_summary = diag_summary.loc[[w for w in lc.KEY_META if w in df_diag["wave"].unique()]]
diag_summary

In [ ]:
corr_rows = []
for wave in lc.KEY_META:
    g = df_diag[df_diag["wave"] == wave]
    if len(g) < 10:
        continue
    corr_rows.append({
        "Wave": wave, "n_beats": len(g),
        "Pearson r":   g["spread_ms"].corr(g["abs_err_ms"], method="pearson"),
        "Spearman rho": g["spread_ms"].corr(g["abs_err_ms"], method="spearman"),
    })
df_corr = pd.DataFrame(corr_rows).set_index("Wave")
df_corr.round(3)

In [ ]:
fig = go.Figure()
for wave in lc.KEY_META:
    g = df_diag[df_diag["wave"] == wave]
    if len(g) < 5:
        continue
    group = lc.KEY_META[wave][0]
    fig.add_trace(go.Scatter(
        x=g["spread_ms"], y=g["abs_err_ms"], mode="markers", name=wave,
        marker=dict(size=5, opacity=0.35, color=lc.GROUP_COLOR[group]),
    ))
fig.update_layout(
    template="plotly_dark", height=520,
    title="Inter-Lead-GT-Spread vs. Detektionsfehler pro Beat (Hybrid-Algorithmus, Test-Holdout)",
    xaxis_title="Inter-Lead-Spread der GT (SD ueber Leads, ms)",
    yaxis_title="Fehler zur gepoolten Konsens-GT (ms, absolut)",
)
fig.show()

In [ ]:
def _spread_bin(g, n_bins=5):
    try:
        return pd.qcut(g["spread_ms"], n_bins, labels=False, duplicates="drop")
    except ValueError:
        return pd.Series(0, index=g.index)

df_diag["spread_bin"] = df_diag.groupby("wave", group_keys=False).apply(_spread_bin)
binned = (df_diag.groupby(["wave", "spread_bin"])
          .agg(mean_spread_ms=("spread_ms", "mean"), mean_abs_err_ms=("abs_err_ms", "mean"), n=("abs_err_ms", "count"))
          .reset_index())

fig = go.Figure()
for wave in lc.KEY_META:
    g = binned[binned["wave"] == wave].sort_values("mean_spread_ms")
    if len(g) < 2:
        continue
    group = lc.KEY_META[wave][0]
    fig.add_trace(go.Scatter(
        x=g["mean_spread_ms"], y=g["mean_abs_err_ms"], mode="lines+markers", name=wave,
        line=dict(color=lc.GROUP_COLOR[group]),
    ))
fig.update_layout(
    template="plotly_dark", height=480,
    title="Gebinnt: mittlerer Detektionsfehler je Spread-Quintil (5 Bins pro Wellentyp, Hybrid, Test-Holdout)",
    xaxis_title="Mittlerer Inter-Lead-Spread im Bin (ms)",
    yaxis_title="Mittlerer Detektionsfehler im Bin (ms)",
)
fig.show()

---

### Zusammenfassung Teil 3

- **T_off-Fix**: RF-CV-Gewinn fuer T_off war nicht auf dem Test-Holdout reproduzierbar (-7.23pp) -- `rf_wins["T_off"]` explizit auf `False` gesetzt, T_off bleibt deterministisch getunt (Teil 1, 66.56% F1).
- **Analyse 1 (Pooled-Konsens-GT)**: siehe `pooled_comparison`-Tabelle und Grafik oben -- traegt die Per-Lead- vs. Pooled-F1-Differenz das von der Hypothese vorhergesagte Muster (QRS ~unveraendert, P/T deutlich hoeher unter Pooled)?
- **Analyse 2 (Spread-vs-Fehler-Korrelation)**: siehe `df_corr` und die beiden Grafiken oben -- positive Pearson/Spearman-Korrelation, v.a. bei P/T-Markern, stuetzt die GT-Unsicherheits-Erklaerung.
- Ergebnisse & Interpretation nach dem ersten Lauf bitte in `docs/ludb_projection_blindness_findings.md` nachtragen (dort auch der volle Kontext der Hypothese und aller bisherigen Zahlen fuer die spaetere Papier-Verwendung).

### Bibliotheks-Uebernahme (weiterhin offen)

Unveraendert gegenueber Teil 2: `vcgsuite` selbst ist nirgends angetastet. Entscheidung zur Uebernahme (nur deterministisch getunt, oder auch RF-Hybrid mit den jetzt korrigierten T_off-Einstellungen) steht weiterhin aus.

### Ergaenzung: Liegt die Detektion innerhalb der Lead-Bandbreite?

Die reine Korrelation (oben) hat ein Confound-Problem: Beats, bei denen sich die 12 Leads uneinig sind, koennten schlicht *insgesamt schwierigere* Beats sein (schwaches Signal, Rauschen, Arrhythmie) -- dann waeren sowohl GT-Spread als auch Algorithmus-Fehler hoch, OHNE dass das etwas ueber "wer hat recht" aussagt. Ein schaerferer Test: liegt die Detektion innerhalb von `[min(Lead-Zeiten), max(Lead-Zeiten)]` -- also innerhalb der Bandbreite, die die 12 Leads selbst aufspannen? Das waere eine plausible "13. Lead-Meinung". Eine Detektion AUSSERHALB dieser Bandbreite kann nicht durch Lead-Unsicherheit erklaert werden -- das ist dann eher ein eigener algorithmischer Fehler. Zusaetzlich der VORZEICHENBEHAFTETE Fehler (`err_ms`), um systematische Verzerrungen (z. B. durchgehend zu spaet) von zufaelligem Rauschen zu unterscheiden -- das ist besonders fuer T_off relevant, siehe Teil 4 unten.

In [ ]:
envelope_summary = df_diag.groupby("wave").agg(
    n=("in_envelope", "count"),
    pct_in_envelope=("in_envelope", "mean"),
    mean_signed_err_ms=("err_ms", "mean"),
    median_signed_err_ms=("err_ms", "median"),
)
envelope_summary["pct_in_envelope"] = (envelope_summary["pct_in_envelope"] * 100).round(1)
envelope_summary = envelope_summary.round(2).loc[[w for w in lc.KEY_META if w in df_diag["wave"].unique()]]
envelope_summary

---

## Teil 4: T_off-Diagnostik — welcher Punkt wird da eigentlich markiert?

Bei der gewaehlten Variante A (Pass 3: CV-Match-Rate 59.1% vs. 49.7% fuer Variante B) bleibt T_off weiterhin ueber `T_turn2` verankert (unveraendert aus `hierarchical.py`, keine eigene LUDB-GT verfuegbar). Neu getunt wurde nur das Fenster (**+20/+100ms** statt vorher +60/+140ms) und die Top-Feature-Strategie: **`(r_lmax, MAX)`**.

`r_lmax` ist gemaess `extract_features()` (`vcgsuite/annotation/features.py`) ein **rollierendes lokales Maximum** von `r` (geglaettet, ±20ms-Fenster). Den Zeitpunkt zu waehlen, an dem `r_lmax` im Fenster maximal ist, heisst praktisch: **den Zeitpunkt der hoechsten lokalen r-Spitze finden** — nicht den Punkt, an dem r zur isoelektrischen Linie zurueckkehrt (klassische "T-Wellen-Ende"-Definition).

**Bugfix vor dieser Diagnose:** die erste Version von `per_beat_diagnostics()` ordnete jedem Beat die naechstgelegene VERFUEGBARE Detektion im ganzen Record zu (aus einer flachen Sample-Liste, keine Beat-Identitaet). Wenn die EIGENE T_off-Detektion eines Beats fehlgeschlagen war (NaN), "erbte" der Beat faelschlich die Detektion eines Nachbar-Beats -- genau das zeigten die von dir hochgeladenen Beispiele (Record 70: drei verschiedene Beats mit fast identischem Fehler, weil sie sich dieselbe geliehene Detektion teilten). `per_beat_diagnostics()` nutzt jetzt `df_beats` direkt (eine Zeile je Beat) statt der flachen Liste -- fehlt die eigene Detektion, wird der Beat uebersprungen statt mit einer fremden ersetzt.

Ausserdem: statt der eigenen einfachen `r(t)`-Kurve mit vertikalen Linien nutzen wir jetzt die **bestehenden Bibliotheks-Plotfunktionen** (`vcgsuite.viz.build_plot_data` / `plot_signal_segments_and_markers` / `plot_vcg_3d`) -- dieselben, die auch in den Beispiel-Notebooks verwendet werden. Die zeigen P/QRS/T-Segmente farbig und alle 12 Marker des Algorithmus auf einen Blick (Signalplot UND 3D-VCG-Trajektorie), wir ergaenzen nur die 12 Lead-GT-T_off-Zeitpunkte als gepunktete Linien obendrauf. Ausgewaehlt werden **3 Beats mit dem groessten Fehler UND 3 "typische" Beats** (Fehler nahe am Median) -- damit wir nicht nur pathologische Ausreisser sehen, sondern auch den Normalfall.

In [ ]:
print("T_off Konsens-Strategien (getunt, Variante", T_OFF_VARIANT, "):", tuned_T_off["strategies"])
print("T_off Fenster (getunt):", tuned_T_off["lo_ms"], "/", tuned_T_off["hi_ms"], "ms relativ zu T_turn2")

df_toff = df_diag[df_diag["wave"] == "T_off"].sort_values("abs_err_ms", ascending=False).reset_index(drop=True)
worst = df_toff.head(3).assign(kind="worst")
med = df_toff["abs_err_ms"].median()
typical = (df_toff.iloc[(df_toff["abs_err_ms"] - med).abs().sort_values().index[:3]]
           .assign(kind="typical"))
example_toff = pd.concat([worst, typical]).reset_index(drop=True)
example_toff[["kind", "record_id", "beat_id", "beat_t", "spread_ms", "err_ms", "abs_err_ms", "in_envelope"]]

In [ ]:
from vcgsuite.viz import build_plot_data, plot_signal_segments_and_markers, plot_vcg_3d

GT_OVERLAY_COLOR = "#00e5ff"  # Cyan -- bewusst ausserhalb der Segment-Farbpalette (P/QRS/T/PQ/ST)


def plot_t_off_example(rid: int, beat_id: int, kind: str, pre_s: float = 0.4, post_s: float = 0.6):
    """Nutzt die bestehenden vcgsuite.viz-Plotfunktionen (Signalplot + 3D-VCG,
    inkl. farbiger P/QRS/T-Segmente und aller 12 Algorithmus-Marker) und
    ergaenzt die 12 Lead-GT-T_off-Zeitpunkte (gepunktet cyan) sowie deren
    Median (durchgezogen cyan) zum direkten visuellen Vergleich."""
    df_b, df_a = _diag_cache[rid]
    row = df_b[df_b["beat_id"] == beat_id].iloc[0]
    r_peak_t = float(row["t_R_peak+"])
    t_start, t_end = r_peak_t - pre_s, r_peak_t + post_s

    df_markers, df_segments = build_plot_data(df_b, beat_id=beat_id, mode="full")

    gt_by_lead = {lead: lc.parse_ludb_annotations(rid, lead) for lead in LEADS}
    gt_t_off = lc.per_lead_times_for_beat(gt_by_lead, r_peak_t, "T_off")

    fig = plot_signal_segments_and_markers(
        df_a, signal_col="A_abs", df_segments=df_segments, df_markers=df_markers,
        t_start=t_start, t_end=t_end,
        title=f"[{kind}] Record {rid}, Beat {beat_id} — A_abs (cyan gepunktet: Lead-GT T_off, cyan durchgezogen: Median)",
    )
    for gt in gt_t_off:
        fig.add_vline(x=gt, line=dict(color=GT_OVERLAY_COLOR, dash="dot", width=1))
    if gt_t_off:
        fig.add_vline(x=float(np.median(gt_t_off)), line=dict(color=GT_OVERLAY_COLOR, width=2))
    fig.show()

    fig3d = plot_vcg_3d(
        df_a, df_segments=df_segments, df_markers=df_markers,
        t_start=t_start, t_end=t_end,
        title=f"[{kind}] Record {rid}, Beat {beat_id} — VCG 3D-Trajektorie",
    )
    fig3d.show()


for _, r in example_toff.iterrows():
    plot_t_off_example(int(r["record_id"]), int(r["beat_id"]), r["kind"])

### Interpretationsleitfaden fuer die Plots oben

- **"worst"-Beispiele:** die 3 Beats mit dem groessten Fehler NACH dem Bugfix (also echte Faelle, keine geliehenen Detektionen mehr). Wenn die gelbe/rote T_off-Markierung (aus `build_plot_data`, Teil des `T`-Segments) dort durchgehend deutlich nach dem cyanen GT-Cluster liegt, auf einer erkennbaren zweiten lokalen Spitze von `A_abs`/r → bestaetigt den Verdacht (U-Welle oder zweite Spitze einer gekerbten/biphasischen T-Welle). Prueft dasselbe auch im 3D-Plot: liegt der T_off-Marker auf einer zweiten kleinen Schlaufe/Ausbeulung der Trajektorie, nachdem die Haupt-T-Schlaufe schon zum Ursprung zurueckgekehrt ist?
- **"typical"-Beispiele:** zeigen den Normalfall (Fehler nahe am Median, 33ms). Wenn hier die Detektion nur leicht neben dem GT-Cluster liegt (nicht auf einer offensichtlich falschen Struktur) → der mittlere Fehler ist eher generische Ungenauigkeit als ein systematisches Falsch-Feature-Problem, und die "worst"-Faelle waeren dann eher Einzelfaelle (z. B. besonders flache/biphasische T-Wellen) statt der Regel.
- Die `envelope_summary`-Tabelle (oben, jetzt mit korrigierten Daten) liefert dasselbe Signal quantitativ: ein niedriger `pct_in_envelope`-Wert UND ein deutlich von 0 verschiedener `mean_signed_err_ms` fuer T_off (im Vergleich zu QRS/P) waere die zahlenbasierte Bestaetigung des visuellen Befunds.
- Falls bestaetigt: Pass 3 fuer T_off nochmal laufen lassen, aber (a) das Fenster enger fassen (z. B. `hi_ms` auf ~60-70ms begrenzen), und/oder (b) alle `*_lmax`-Features aus `CANDIDATE_FEATURES` fuer T_off ausschliessen und stattdessen gezielt nach einem "r faellt unter Schwelle X"-Feature suchen.

---

## Teil 5: Experimentelle U-Wellen-Erkennung (Bonus, unvalidiert)

Der Verdacht aus Teil 4/`ludb_projection_reconstruction.ipynb`: `(r_lmax, MAX)` im alten T_off-Fenster hat
vermutlich systematisch eine U-Welle statt des echten T-Endes erwischt (+130ms-Bias, plausibel im
T-U-Intervall). Was fuer T_off ein Fehler war, ist als eigenstaendiger Marker eine sinnvolle Idee -- also
genau das wiederverwenden, nur korrekt benannt und mit einer Praesenz-Pruefung (U-Wellen sind nicht in
jedem Beat/Lead sichtbar, anders als P/QRS/T).

**Wichtige Einschraenkung, die diesen Abschnitt fundamental von allen anderen unterscheidet: LUDB hat KEINE
U-Wellen-Ground-Truth** (die WFDB-Annotationssymbole kennen nur `p`/`N`/`t` fuer P-/QRS-/T-Welle, siehe
Schritt 2 in `ludb_validation.ipynb`). Es gibt also **kein Zielsignal fuer einen Grid-Search** wie bei den
anderen 7 Markern -- dieser Abschnitt ist rein heuristisch, nicht CV-getunt, nicht quantitativ validiert.
Nutzerzitat zur Erwartungshaltung: "auch wenn es etwas fehlerhaft ist" -- genau das ist hier der Stand:
best-effort, zur Inspektion gedacht, NICHT fuer eine Uebernahme in `vcgsuite` ohne weitere Pruefung.

In [ ]:
def detect_u_wave_naive(df_analysis: pd.DataFrame, t_off: float, t_peak_r: float,
                        lo_ms: float = 0.0, hi_ms: float = 250.0,
                        min_rel_amp: float = 0.03, max_rel_amp: float = 0.50,
                        local_window_ms: float = 20.0) -> dict:
    """EXPERIMENTELLER, UNGETUNTER U-Wellen-Detektor (siehe Markdown oben --
    keine LUDB-GT verfuegbar). Sucht im Fenster [t_off+lo_ms, t_off+hi_ms]
    nach einer lokalen r-Spitze (dieselbe r_lmax-Feature-Familie, die wir aus
    T_off entfernt haben, Pass 3b). Akzeptiert den Fund nur als plausible
    U-Welle, wenn seine Amplitude relativ zur T-Wellen-Amplitude im
    literaturtypischen Bereich liegt (deutlich kleiner als die T-Welle) --
    sonst explizit NaN ('keine U-Welle gefunden') statt immer etwas zu
    erzwingen. `t_peak_r` = Zeitpunkt des T-Wellen-Peaks (z. B. der bereits
    per _nearest_r() gewaehlte T_turn1/T_turn2, wie in beats_to_sample_dict()).
    """
    if np.isnan(t_off) or np.isnan(t_peak_r) or "r" not in df_analysis.columns:
        return {"U_peak": np.nan, "U_amp_rel": np.nan}

    fs = df_analysis.attrs.get("fs", 500.0)
    t = df_analysis["Time"].to_numpy()
    r = df_analysis["r"].to_numpy()

    idx_off = int(min(max(np.searchsorted(t, t_off), 0), len(r) - 1))
    baseline = float(r[idx_off])

    idx_tpeak = int(min(max(np.searchsorted(t, t_peak_r), 0), len(r) - 1))
    t_wave_amp = abs(float(r[idx_tpeak]) - baseline)
    if t_wave_amp < 1e-9:
        return {"U_peak": np.nan, "U_amp_rel": np.nan}

    lo_idx = int(np.searchsorted(t, t_off + lo_ms / 1000.0))
    hi_idx = int(min(np.searchsorted(t, t_off + hi_ms / 1000.0), len(r)))
    if hi_idx - lo_idx < 5:
        return {"U_peak": np.nan, "U_amp_rel": np.nan}

    lw = max(1, int(local_window_ms / 1000.0 * fs))
    window = r[lo_idx:hi_idx]
    r_lmax = np.array([window[max(0, i - lw):min(len(window), i + lw + 1)].max()
                       for i in range(len(window))])
    cand_idx = int(np.argmax(r_lmax))
    u_amp = abs(float(window[cand_idx]) - baseline)
    rel_amp = u_amp / t_wave_amp

    if min_rel_amp <= rel_amp <= max_rel_amp:
        return {"U_peak": float(t[lo_idx + cand_idx]), "U_amp_rel": rel_amp}
    return {"U_peak": np.nan, "U_amp_rel": rel_amp}

In [ ]:
u_rows = []
for rid, (df_b, df_a) in _diag_cache.items():
    fs = df_a.attrs["fs"]
    for _, brow in df_b.iterrows():
        t_off = brow.get("t_T_off", np.nan)
        t1, t2 = brow.get("t_T_turn1", np.nan), brow.get("t_T_turn2", np.nan)
        cands = [tt for tt in (t1, t2) if not np.isnan(tt)]
        if not cands:
            continue
        if len(cands) == 1:
            t_peak_r = cands[0]
        else:
            r1, r2 = lc._nearest_r(df_a, t1, fs), lc._nearest_r(df_a, t2, fs)
            t_peak_r = t1 if r1 >= r2 else t2
        res = detect_u_wave_naive(df_a, t_off, t_peak_r)
        u_rows.append({"record_id": rid, "beat_id": int(brow["beat_id"]), "t_off": t_off,
                       "U_peak": res["U_peak"], "U_amp_rel": res["U_amp_rel"]})

df_u = pd.DataFrame(u_rows)
n_detected = int(df_u["U_peak"].notna().sum())
print(f"{len(df_u)} Beats geprueft (Test-Holdout), bei {n_detected} "
      f"({n_detected / len(df_u):.1%}) eine plausible U-Welle gefunden "
      f"(Amplitude 3-50% der T-Wellen-Amplitude).")

df_u["U_peak_rel_to_Toff_ms"] = (df_u["U_peak"] - df_u["t_off"]) * 1000.0
df_u.loc[df_u["U_peak"].notna(), ["U_peak_rel_to_Toff_ms", "U_amp_rel"]].describe().round(3)

In [ ]:
# Zufallsstichprobe aus den Beats mit erkannter U-Welle -- rein zur visuellen
# Plausibilitaetspruefung, da keine GT existiert. build_plot_data/plot_signal_segments_and_markers
# kennen 'U_peak' nicht als eigenen Markertyp (nicht in vcgsuite.viz.vcg_annotated.GT_MARKERS --
# bewusst nicht geaendert, siehe Hinweis oben), daher wird die U-Welle manuell als vertikale
# Linie ergaenzt statt ueber build_plot_data() eingespeist.
example_u = df_u[df_u["U_peak"].notna()].sample(n=min(6, n_detected), random_state=42).reset_index(drop=True)


def plot_u_wave_example(rid: int, beat_id: int, u_peak_t: float, pre_s: float = 0.4, post_s: float = 0.7):
    df_b, df_a = _diag_cache[rid]
    row = df_b[df_b["beat_id"] == beat_id].iloc[0]
    r_peak_t = float(row["t_R_peak+"])
    t_start, t_end = r_peak_t - pre_s, r_peak_t + post_s

    df_markers, df_segments = build_plot_data(df_b, beat_id=beat_id, mode="full")
    fig = plot_signal_segments_and_markers(
        df_a, signal_col="r", df_segments=df_segments, df_markers=df_markers,
        t_start=t_start, t_end=t_end,
        title=f"Record {rid}, Beat {beat_id} — r(t), vermutete U-Welle (magenta gestrichelt)",
    )
    fig.add_vline(x=u_peak_t, line=dict(color="#e83e8c", dash="dash", width=2))
    fig.show()


for _, r in example_u.iterrows():
    plot_u_wave_example(int(r["record_id"]), int(r["beat_id"]), r["U_peak"])

### Interpretationsleitfaden

- **Erkennungsrate (`n_detected`):** sollte eine Minderheit der Beats sein (Literatur: U-Wellen sind nicht
  immer sichtbar, haeufiger bei langsamerer Herzfrequenz/in bestimmten Ableitungen). Nahe 0% heisst, die
  Schwellenwerte sind zu streng; nahe 100% heisst, es wird fast immer irgendetwas gefunden (Schwellen zu
  locker, evtl. wird wieder systematisch derselbe Strukturtyp wie vorher bei T_off erwischt, nur jetzt als
  'U-Welle' labelt statt als T_off -- kein echter Fortschritt).
- **`U_amp_rel`:** sollte deutlich unter 1.0 clustern (U-Welle klar kleiner als T-Welle). Werte nahe an 0.5
  (der harten Obergrenze) sind ein Warnsignal fuer Fehlklassifikation (evtl. eine zweite, fast T-Wellen-grosse
  Struktur -- z. B. eine echte zweite T-Welle bei Extrasystolen -- keine U-Welle).
- **`U_peak_rel_to_Toff_ms`:** Median/Verteilung geben eine grobe Idee, ob die gefundene Struktur zeitlich
  zum bekannten T-U-Intervall passt.
- **Plots:** die einzig wirklich aussagekraeftige Pruefung ohne GT -- sieht die magenta Linie in den Plots
  optisch nach einer plausiblen U-Welle aus (kleine, separate Erhebung nach dem Signal-Ende der T-Welle) oder
  nach Rauschen/einem Artefakt?

**Empfehlung, falls das grundsaetzlich brauchbar aussieht:** vorerst NICHT in `vcgsuite` uebernehmen (keine
Validierungsbasis) -- stattdessen (a) auf eine groessere/zufaellige Stichprobe ausweiten und manuell durchklicken,
und/oder (b) pruefen, ob ein anderes oeffentliches Dataset U-Wellen-Annotationen hat, die sich fuer eine echte
Validierung nutzen liessen. Bis dahin bleibt das ein reines Notebook-Experiment, klar als 'nicht validiert'
gekennzeichnet -- passend zur Erwartungshaltung ('etwas fehlerhaft').